In [ ]:
# ============================================================
# GOVERNANCE + STRATEGY SECTION GENERATORS
# Role-based Azure OpenAI REST endpoints
# Writer: GPT-5.1 | Judge: GPT-5.2 (LLM-only evaluation) | Reviser: GPT-4.1
# ============================================================

import os
import json
import re
import urllib.request
import urllib.error
import time
import random
from typing import TypedDict, Literal
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langgraph.graph import StateGraph, START, END

# ── ENV LOADING ──────────────────────────────────────────────
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")

# Shared-key fallback.
# When all three deployments belong to the same Azure resource, one Azure key
# can authenticate all roles. The code also accepts any existing role-specific
# key as the shared fallback, which prevents unnecessary configuration failures.
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

_shared_key_fallback = (
    AZURE_OPENAI_API_KEY
    or os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or os.getenv("AZURE_OPENAI_REVISER_API_KEY")
)

AZURE_OPENAI_WRITER_API_KEY = (
    os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or _shared_key_fallback
)
AZURE_OPENAI_JUDGE_API_KEY = (
    os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or _shared_key_fallback
)
AZURE_OPENAI_REVISER_API_KEY = (
    os.getenv("AZURE_OPENAI_REVISER_API_KEY")
    or _shared_key_fallback
)

# Full Azure chat-completions deployment URLs.
# AZURE_OPENAI_CHAT_URL is retained as a backward-compatible writer fallback.
AZURE_OPENAI_WRITER_URL = (
    os.getenv("AZURE_OPENAI_WRITER_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
)
AZURE_OPENAI_JUDGE_URL = os.getenv("AZURE_OPENAI_JUDGE_URL")
AZURE_OPENAI_REVISER_URL = os.getenv("AZURE_OPENAI_REVISER_URL")


def _clean_url(value: str | None) -> str | None:
    if not value:
        return None
    return value.strip().strip('"').strip("'")


AZURE_OPENAI_WRITER_URL = _clean_url(AZURE_OPENAI_WRITER_URL)
AZURE_OPENAI_JUDGE_URL = _clean_url(AZURE_OPENAI_JUDGE_URL)
AZURE_OPENAI_REVISER_URL = _clean_url(AZURE_OPENAI_REVISER_URL)


def validate_role_config() -> None:
    required = {
        "AZURE_OPENAI_WRITER_API_KEY": AZURE_OPENAI_WRITER_API_KEY,
        "AZURE_OPENAI_JUDGE_API_KEY": AZURE_OPENAI_JUDGE_API_KEY,
        "AZURE_OPENAI_REVISER_API_KEY": AZURE_OPENAI_REVISER_API_KEY,
        "AZURE_OPENAI_WRITER_URL": AZURE_OPENAI_WRITER_URL,
        "AZURE_OPENAI_JUDGE_URL": AZURE_OPENAI_JUDGE_URL,
        "AZURE_OPENAI_REVISER_URL": AZURE_OPENAI_REVISER_URL,
    }

    missing = [name for name, value in required.items() if not value]
    if missing:
        loaded_flags = {
            "shared_key_loaded": bool(AZURE_OPENAI_API_KEY),
            "writer_key_loaded": bool(AZURE_OPENAI_WRITER_API_KEY),
            "judge_key_loaded": bool(AZURE_OPENAI_JUDGE_API_KEY),
            "reviser_key_loaded": bool(AZURE_OPENAI_REVISER_API_KEY),
            "writer_url_loaded": bool(AZURE_OPENAI_WRITER_URL),
            "judge_url_loaded": bool(AZURE_OPENAI_JUDGE_URL),
            "reviser_url_loaded": bool(AZURE_OPENAI_REVISER_URL),
        }
        raise ValueError(
            "Missing role-based Azure configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded configuration flags (keys are never printed):\n"
            + json.dumps(loaded_flags, indent=2)
            + "\n\nRequired .env configuration when all deployments use the same Azure resource:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_WRITER_URL=<full GPT-5.1 deployment URL>\n"
              "AZURE_OPENAI_JUDGE_URL=<full GPT-5.2 deployment URL>\n"
              "AZURE_OPENAI_REVISER_URL=<full GPT-4.1 deployment URL>\n\n"
              "Use role-specific API keys only when a deployment belongs to a different Azure resource."
        )

    for name, url in {
        "AZURE_OPENAI_WRITER_URL": AZURE_OPENAI_WRITER_URL,
        "AZURE_OPENAI_JUDGE_URL": AZURE_OPENAI_JUDGE_URL,
        "AZURE_OPENAI_REVISER_URL": AZURE_OPENAI_REVISER_URL,
    }.items():
        if not url.startswith("https://"):
            raise ValueError(f"{name} must be a full HTTPS Azure deployment URL: {url!r}")


validate_role_config()

print("Role-based Azure OpenAI configuration loaded")
print("Writer endpoint (GPT-5.1):", AZURE_OPENAI_WRITER_URL[:90] + "...")
print("Judge endpoint  (GPT-5.2):", AZURE_OPENAI_JUDGE_URL[:90] + "...")
print("Reviser endpoint (GPT-4.1):", AZURE_OPENAI_REVISER_URL[:90] + "...")


def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: list[dict],
    max_output_tokens: int,
    json_mode: bool = False,
    temperature: float | None = None,
    use_max_completion_tokens: bool = False,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 4,
) -> dict:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Behaviour:
    - Retries transient 500/502/503/504 and connection errors.
    - For GPT-5.x gateways, first tries `max_completion_tokens`.
    - If the gateway returns 400/500, retries using `max_tokens`, because some
      enterprise proxies do not yet forward `max_completion_tokens` correctly.
    - Does not expose API keys in errors.
    """

    preferred_field = (
        "max_completion_tokens" if use_max_completion_tokens else "max_tokens"
    )
    token_fields = [preferred_field]
    if preferred_field == "max_completion_tokens":
        token_fields.append("max_tokens")

    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
        }

        if temperature is not None:
            payload["temperature"] = temperature

        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": api_key,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {url}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                transient = exc.code in {500, 502, 503, 504}
                compatibility_candidate = (
                    token_field == "max_completion_tokens"
                    and exc.code in {400, 422, 500}
                )

                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: server error {exc.code}; "
                        f"retrying attempt {attempt + 1}/{max_attempts} "
                        f"in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support "
                        "`max_completion_tokens`; retrying with `max_tokens`."
                    )
                    break

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {url!r}\n"
                    f"Error: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(
        f"{request_label} request failed for an unknown reason."
    )

def _extract_message_content(data: dict) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )
    return content


def call_writer_llm(system_prompt: str, user_prompt: str) -> str:
    """GPT-5.1 writer."""
    data = _azure_chat_completion(
        url=AZURE_OPENAI_WRITER_URL,
        api_key=AZURE_OPENAI_WRITER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2400,
        use_max_completion_tokens=True,
        temperature=None,
        request_label="GPT-5.1 writer",
    )
    return _extract_message_content(data)


def _extract_json_object(text: str) -> str:
    """
    Extract the outermost JSON object from model output.
    Handles accidental markdown fences or leading/trailing commentary.
    """
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")
    if first >= 0 and last > first:
        return text[first:last + 1]

    return text


def call_judge_llm_json(system_prompt: str, user_prompt: str) -> dict:
    """
    GPT-5.2 judge with JSON-safe retry logic.

    First call:
    - Requests valid JSON mode.
    - Uses a larger output allowance to avoid truncation.

    On invalid/truncated JSON:
    - Sends the returned content back to GPT-5.2 for JSON repair.
    - Requests a concise, complete JSON object only.
    """
    data = _azure_chat_completion(
        url=AZURE_OPENAI_JUDGE_URL,
        api_key=AZURE_OPENAI_JUDGE_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2800,
        use_max_completion_tokens=True,
        temperature=None,
        json_mode=True,
        request_label="GPT-5.2 judge",
    )

    content = _extract_message_content(data)
    candidate = _extract_json_object(content)

    try:
        return json.loads(candidate)

    except json.JSONDecodeError:
        print("GPT-5.2 judge returned incomplete/invalid JSON. Attempting JSON repair...")

        repair_system = (
            "You repair malformed or truncated JSON. "
            "Return one complete valid JSON object only. "
            "Preserve the original meaning, scores, checklist values, issues, and fixes. "
            "Keep strings concise. Do not add markdown fences or commentary."
        )

        repair_user = f"""
Repair the following malformed or truncated judge output into one complete valid JSON object.

Requirements:
- Keep the same top-level fields when present.
- Finish incomplete strings and arrays conservatively.
- Limit each issue/fix string to at most 35 words.
- Limit arrays to the 6 most important items.
- Return JSON only.

MALFORMED OUTPUT:
{content}
""".strip()

        repaired_data = _azure_chat_completion(
            url=AZURE_OPENAI_JUDGE_URL,
            api_key=AZURE_OPENAI_JUDGE_API_KEY,
            messages=[
                {"role": "system", "content": repair_system},
                {"role": "user", "content": repair_user},
            ],
            max_output_tokens=2400,
            use_max_completion_tokens=True,
            temperature=None,
            json_mode=True,
            request_label="GPT-5.2 judge JSON repair",
        )

        repaired_content = _extract_message_content(repaired_data)
        repaired_candidate = _extract_json_object(repaired_content)

        try:
            return json.loads(repaired_candidate)
        except json.JSONDecodeError as exc:
            raise ValueError(
                "GPT-5.2 judge failed to return valid JSON even after repair.\n"
                f"Original output preview:\n{content[:4000]}\n\n"
                f"Repair output preview:\n{repaired_content[:4000]}"
            ) from exc


def call_reviser_llm(system_prompt: str, user_prompt: str) -> str:
    """GPT-4.1 reviser."""
    data = _azure_chat_completion(
        url=AZURE_OPENAI_REVISER_URL,
        api_key=AZURE_OPENAI_REVISER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=2400,
        use_max_completion_tokens=False,
        temperature=0.1,
        request_label="GPT-4.1 reviser",
    )
    return _extract_message_content(data)


print("Role-specific LLM helper functions ready")


## JSON-safety update

- Increased Writer and Reviser output limits to reduce incomplete sections.
- Increased GPT-5.2 Judge output allowance.
- Added automatic JSON extraction and one repair retry when the Judge returns truncated or malformed JSON.
- Judge prompts now request concise issue/fix arrays to reduce output truncation.


In [ ]:
# ── OPTIONAL ROLE ENDPOINT SMOKE TEST ────────────────────────
# Run this cell before invoking the LangGraph pipeline.
# It tests each deployment with a very small request, making it easier to
# distinguish endpoint/model problems from prompt-size or graph problems.

def test_role_endpoints() -> dict:
    results = {}

    tests = [
        ("writer_gpt_5_1", call_writer_llm, "Reply with exactly: writer ok"),
        (
            "judge_gpt_5_2",
            lambda s, u: call_judge_llm_json(s, u),
            'Return only this JSON object: {"status":"judge ok"}',
        ),
        ("reviser_gpt_4_1", call_reviser_llm, "Reply with exactly: reviser ok"),
    ]

    for name, fn, prompt in tests:
        try:
            result = fn(
                "You are performing a minimal endpoint connectivity test.",
                prompt,
            )
            results[name] = {
                "success": True,
                "response_preview": str(result)[:200],
            }
        except Exception as exc:
            results[name] = {
                "success": False,
                "error": str(exc)[:1000],
            }

    print(json.dumps(results, indent=2, ensure_ascii=False))
    return results


# Uncomment to test all three deployments before running the graphs:
# endpoint_test_results = test_role_endpoints()


In [ ]:
# ── LOAD SECTION-SPECIFIC PAYLOADS ───────────────────────────
# The report-generation notebook should consume the payloads produced by
# the data-prep notebook:
#   payload_BANK01_governance.json
#   payload_BANK01_strategy.json
#
# A full payload fallback is kept only for local debugging.

def find_payload_file(filename: str) -> Path | None:
    search_dirs = [
        Path.cwd(),
        Path.cwd() / "Data",
        Path.cwd() / "payloads",
        Path.cwd().parent / "payloads",
        Path.cwd().parent / "Data",
        Path("/mnt/data"),  # useful in this ChatGPT sandbox only
    ]
    for base in search_dirs:
        candidate = base / filename
        if candidate.exists():
            return candidate
    return None


def load_json_file(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


GOVERNANCE_PAYLOAD_PATH = (
    find_payload_file("payload_BANK01_governance.json")
    or find_payload_file("payload_BANK01.json")
)

STRATEGY_PAYLOAD_PATH = (
    find_payload_file("payload_BANK01_strategy.json")
    or find_payload_file("payload_BANK01.json")
)

if GOVERNANCE_PAYLOAD_PATH is None:
    raise FileNotFoundError(
        "Could not find payload_BANK01_governance.json or payload_BANK01.json."
    )

if STRATEGY_PAYLOAD_PATH is None:
    raise FileNotFoundError(
        "Could not find payload_BANK01_strategy.json or payload_BANK01.json."
    )

governance_payload = load_json_file(GOVERNANCE_PAYLOAD_PATH)
strategy_payload = load_json_file(STRATEGY_PAYLOAD_PATH)

# PATCH: add climate_risk_register to Governance evidence if it is absent there
# but available in the Strategy payload. This avoids under-disclosing IFRS S2
# management responsibility when risk-register evidence exists elsewhere in the
# prepared section payloads.
if "climate_risk_register" not in governance_payload and "climate_risk_register" in strategy_payload:
    governance_payload["climate_risk_register"] = strategy_payload["climate_risk_register"]
    print("Patched Governance payload: added climate_risk_register from Strategy payload.")

# Backward-compatible alias used by older cells.
payload = governance_payload
PAYLOAD_PATH = GOVERNANCE_PAYLOAD_PATH

bank_name = governance_payload["bank"]["bank_name"]

print(f"Loaded Governance payload for: {bank_name}")
print(f"Governance payload path: {GOVERNANCE_PAYLOAD_PATH}")
print(f"Governance top-level keys: {list(governance_payload.keys())}")
print("Governance has climate_risk_register:", "climate_risk_register" in governance_payload)

print(f"\nLoaded Strategy payload for: {strategy_payload['bank']['bank_name']}")
print(f"Strategy payload path: {STRATEGY_PAYLOAD_PATH}")
print(f"Strategy top-level keys: {list(strategy_payload.keys())}")
print("Strategy has climate_risk_register:", "climate_risk_register" in strategy_payload)

In [ ]:
# ── GOVERNANCE EVIDENCE EXTRACTOR ────────────────────────────
# Data-aware behaviour:
# - The Governance payload is the source of Governance facts.
# - This uploaded Governance payload contains governance, board_minutes and reporting_kpis.
# - It does not always contain climate_risk_register. If risk-register rows are absent,
#   the Management Responsibility subsection must be limited to the management committee,
#   board reporting frequency, ERM integration flag and major-transaction climate check.
# - The writer must not invent formal committee charters, trade-offs, escalation thresholds,
#   skills adequacy assessments, or assurance over financed emissions.

def _is_present(value) -> bool:
    return value is not None and str(value).strip().lower() not in {"", "nan", "none", "null"}


def _safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def _normalise_text(value) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def _format_reporting_frequency(value: str | None) -> str | None:
    if not _is_present(value):
        return None
    return str(value).replace("_", "-").lower()


def extract_management_process_evidence(payload: dict, year: int = 2024) -> dict:
    """
    Summarise Management Responsibility evidence.

    Preferred source: climate_risk_register, when present.
    Fallback source: governance table fields in the Governance payload.
    """
    risks = [
        r for r in payload.get("climate_risk_register", [])
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
    ]

    gov_records = payload.get("governance", [])
    gov_2024 = next(
        (
            r for r in gov_records
            if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
        ),
        {},
    )

    governance_controls_available = any(
        _is_present(gov_2024.get(field))
        for field in [
            "management_committee_name",
            "climate_risk_reporting_to_board",
            "erm_integration_flag",
            "major_transactions_climate_check",
        ]
    )

    if not risks:
        return {
            "risk_register_available": False,
            "governance_controls_available": governance_controls_available,
            "reporting_year": year,
            "management_committee_name": gov_2024.get("management_committee_name"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "formal_escalation_thresholds_available": False,
            "message": (
                "The available Governance evidence does not contain climate_risk_register records. "
                "Management Responsibility can be described only using governance-level evidence."
            ),
            "process_flow_instruction": (
                "Do not claim a risk-register workflow, risk counts, risk categories, scenario links, "
                "monitoring frequencies by risk, or mitigation actions unless risk-register evidence is present. "
                "Use the available governance controls only: management committee name, board reporting frequency, "
                "ERM integration flag and major-transaction climate check."
            ),
        }

    frequencies = sorted({
        str(r.get("monitoring_frequency"))
        for r in risks
        if _is_present(r.get("monitoring_frequency"))
    })
    risk_categories = sorted({
        str(r.get("risk_category"))
        for r in risks
        if _is_present(r.get("risk_category"))
    })
    risk_ratings = sorted({
        str(r.get("risk_rating"))
        for r in risks
        if _is_present(r.get("risk_rating"))
    })
    scenario_links = sorted({
        str(r.get("scenario_analysis_link"))
        for r in risks
        if _is_present(r.get("scenario_analysis_link"))
    })
    mitigation_actions = sorted({
        str(r.get("mitigation_actions"))
        for r in risks
        if _is_present(r.get("mitigation_actions"))
    })

    integrated_count = sum(1 for r in risks if r.get("erm_integrated_flag") is True)
    changed_count = sum(1 for r in risks if r.get("changed_since_prior_period") is True)

    rating_priority = {"critical": 4, "high": 3, "medium": 2, "low": 1}
    sorted_risks = sorted(
        risks,
        key=lambda r: (
            rating_priority.get(str(r.get("risk_rating", "")).lower(), 0),
            float(r.get("financial_impact_meur") or 0),
        ),
        reverse=True,
    )

    material_risk_examples = []
    for r in sorted_risks[:5]:
        material_risk_examples.append({
            "risk_id": r.get("risk_id"),
            "risk_name": r.get("risk_name"),
            "risk_category": r.get("risk_category"),
            "risk_rating": r.get("risk_rating"),
            "time_horizon": r.get("time_horizon"),
            "financial_impact_meur": r.get("financial_impact_meur"),
            "monitoring_frequency": r.get("monitoring_frequency"),
            "erm_integrated_flag": r.get("erm_integrated_flag"),
            "scenario_analysis_link": r.get("scenario_analysis_link"),
            "mitigation_actions": r.get("mitigation_actions"),
        })

    return {
        "risk_register_available": True,
        "governance_controls_available": governance_controls_available,
        "reporting_year": year,
        "management_committee_name": gov_2024.get("management_committee_name"),
        "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
        "erm_integration_flag": gov_2024.get("erm_integration_flag"),
        "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
        "risk_count": len(risks),
        "erm_integrated_count": integrated_count,
        "changed_since_prior_period_count": changed_count,
        "monitoring_frequencies": frequencies,
        "risk_categories": risk_categories,
        "risk_ratings": risk_ratings,
        "scenario_analysis_links": scenario_links,
        "mitigation_actions": mitigation_actions[:8],
        "material_risk_examples": material_risk_examples,
        "formal_escalation_thresholds_available": False,
        "process_flow_instruction": (
            "Write management responsibility as a process flow only if risk-register evidence is present: "
            "risk identification/register, classification by category/time horizon/rating, monitoring frequency, "
            "scenario links, mitigation actions and ERM integration. Do not invent formal escalation thresholds."
        ),
    }


def extract_governance_evidence(payload: dict) -> dict:
    gov_records = payload.get("governance", [])
    board_minutes = payload.get("board_minutes", [])
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})
    metadata = payload.get("metadata", {})

    reporting_year = int(metadata.get("reporting_year", 2024))

    gov_by_year = {
        str(r["reporting_year"]): r
        for r in gov_records
        if isinstance(r, dict) and "reporting_year" in r
    }

    gov_trend = []
    for year in ["2022", "2023", "2024"]:
        if year in gov_by_year:
            g = gov_by_year[year]
            gov_trend.append({
                "year": int(year),
                "esg_committee_meetings": g.get("esg_committee_meetings_per_year"),
                "board_climate_expertise_pct": g.get("board_climate_expertise_pct"),
                "ceo_esg_compensation_pct": g.get("ceo_esg_compensation_pct"),
                "all_exec_climate_remuneration_pct": g.get("all_exec_climate_remuneration_pct"),
                "climate_on_board_agenda_pct": g.get("climate_on_board_agenda_pct"),
                "management_committee_name": g.get("management_committee_name"),
                "board_full_meeting_frequency": g.get("board_full_meeting_frequency"),
            })

    PRIORITY_TOPICS = [
        "esg report", "scenario analysis", "transition plan", "net-zero",
        "net zero", "target", "carbon credit", "physical risk", "green finance"
    ]

    minutes_2024 = [
        m for m in board_minutes
        if isinstance(m, dict)
        and _safe_int(m.get("reporting_year")) == reporting_year
        and m.get("decision_made_flag") is True
        and _is_present(m.get("decision_summary"))
        and _is_present(m.get("meeting_id"))
    ]

    # Group by decision text so the same decision is not treated as two separate
    # Board/committee decisions when it appears in multiple meeting records.
    grouped_decisions = {}
    for m in minutes_2024:
        key = re.sub(r"\s+", " ", str(m.get("decision_summary", "")).lower().strip())
        if key not in grouped_decisions:
            grouped_decisions[key] = {
                "decision": _normalise_text(m.get("decision_summary")),
                "dates": set(),
                "committees": set(),
                "committee_types": set(),
                "topics_discussed": set(),
                "meeting_ids": set(),
                "ifrs_evidence_paras": set(),
            }

        grouped_decisions[key]["dates"].add(m.get("meeting_date"))
        grouped_decisions[key]["committees"].add(m.get("committee_name"))
        grouped_decisions[key]["committee_types"].add(m.get("committee_type"))
        grouped_decisions[key]["meeting_ids"].add(m.get("meeting_id"))
        grouped_decisions[key]["ifrs_evidence_paras"].add(m.get("ifrs_s2_para_evidence"))
        topics = str(m.get("climate_topics_discussed") or "").split("|")
        grouped_decisions[key]["topics_discussed"].update(t for t in topics if _is_present(t))

    selected_decisions = []
    for item in grouped_decisions.values():
        dates = sorted(x for x in item["dates"] if _is_present(x))
        committees = sorted(x for x in item["committees"] if _is_present(x))
        topics = sorted(x for x in item["topics_discussed"] if _is_present(x))
        meeting_ids = sorted(x for x in item["meeting_ids"] if _is_present(x))
        decision = item["decision"]
        score = sum(1 for term in PRIORITY_TOPICS if term in decision.lower() or term in " ".join(topics).lower())
        selected_decisions.append({
            "primary_date": dates[0] if dates else None,
            "dates": dates,
            "committees": committees,
            "committee_types": sorted(x for x in item["committee_types"] if _is_present(x)),
            "topics_discussed": topics,
            "decision": decision,
            "ifrs_evidence_paras": sorted(x for x in item["ifrs_evidence_paras"] if _is_present(x)),
            "internal_refs": [f"[REF:{mid}]" for mid in meeting_ids],
            "decision_group_score": score,
        })

    selected_decisions = sorted(
        selected_decisions,
        key=lambda d: (-d.get("decision_group_score", 0), d.get("primary_date") or "")
    )[:6]

    gov_2024 = gov_by_year.get(str(reporting_year), {})

    # Conservative evidence-gap assessment.
    governance_instrument_fields = [
        "committee_charter", "committee_terms_of_reference", "board_mandate",
        "esg_committee_mandate", "formal_climate_mandate",
        "governance_policy_reference", "committee_charter_climate_mandate",
    ]
    formal_mandate_available = any(_is_present(gov_2024.get(f)) for f in governance_instrument_fields)

    tradeoff_terms = [
        "tradeoff", "trade-off", "capital allocation", "profitability",
        "implementation cost", "risk appetite", "competing priority", "competing priorities"
    ]
    tradeoff_decisions = [
        m for m in minutes_2024
        if any(
            term in str(m.get("decision_summary", "")).lower()
            or term in str(m.get("climate_topics_discussed", "")).lower()
            for term in tradeoff_terms
        )
    ]
    board_tradeoff_evidence_available = len(tradeoff_decisions) > 0

    skills_process_fields = [
        "skills_matrix", "skills_assessment_process", "board_skills_review",
        "skills_adequacy_assessment", "director_training_frequency",
        "training_hours", "skills_gap_analysis",
    ]
    skills_adequacy_process_available = any(_is_present(gov_2024.get(f)) for f in skills_process_fields)

    assurance_scope = str(gov_2024.get("assurance_scope", ""))
    financed_emissions_2024 = reporting_kpis.get("financed_emissions_2024_tco2e")
    financed_in_scope = "financed" in assurance_scope.lower() or "scope 3" in assurance_scope.lower()

    assurance_scope_limitation = {
        "assurance_scope": assurance_scope,
        "external_assurance": gov_2024.get("external_assurance"),
        "provider": gov_2024.get("assurance_provider"),
        "standard": gov_2024.get("assurance_standard"),
        "financed_emissions_2024_tco2e": financed_emissions_2024,
        "financed_emissions_in_scope": financed_in_scope,
        "instruction": (
            "State that assurance covers only the stated scope. If the stated scope is Scope 1 and 2 emissions, "
            "do not imply financed emissions or other Scope 3 categories are assured. For a bank, explicitly clarify "
            "that financed emissions are outside the stated assurance scope based on available evidence."
        ),
    }

    management_evidence = extract_management_process_evidence(payload, year=reporting_year)

    return {
        "bank": {
            "name": bank.get("bank_name"),
            "bank_id": bank.get("bank_id"),
            "country": bank.get("country"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "reporting_year": reporting_year,
        "comparative_years": metadata.get("comparative_years", [2022, 2023]),
        "payload_profile": {
            "source_payload": "governance",
            "top_level_keys": list(payload.keys()),
            "climate_risk_register_in_payload": "climate_risk_register" in payload,
            "board_minutes_count": len(board_minutes),
            "governance_record_count": len(gov_records),
        },
        "governance_2024": {
            "board_size": gov_2024.get("board_size"),
            "independent_directors_pct": gov_2024.get("independent_directors_pct"),
            "esg_committee_exists": gov_2024.get("esg_committee_exists"),
            "esg_committee_meetings_per_year": gov_2024.get("esg_committee_meetings_per_year"),
            "board_climate_expertise_pct": gov_2024.get("board_climate_expertise_pct"),
            "ceo_compensation_esg_linked": gov_2024.get("ceo_compensation_esg_linked"),
            "ceo_esg_compensation_pct": gov_2024.get("ceo_esg_compensation_pct"),
            "all_exec_climate_remuneration_pct": gov_2024.get("all_exec_climate_remuneration_pct"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "climate_on_board_agenda_pct": gov_2024.get("climate_on_board_agenda_pct"),
            "board_full_meeting_frequency": gov_2024.get("board_full_meeting_frequency"),
            "management_committee_name": gov_2024.get("management_committee_name"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "skills_development_programme": gov_2024.get("skills_development_programme"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "external_assurance": gov_2024.get("external_assurance"),
            "assurance_provider": gov_2024.get("assurance_provider"),
            "assurance_scope": gov_2024.get("assurance_scope"),
            "assurance_standard": gov_2024.get("assurance_standard"),
            "tcfd_aligned": gov_2024.get("tcfd_aligned"),
            "ifrs_s2_aligned": gov_2024.get("ifrs_s2_aligned"),
        },
        "governance_trend": gov_trend,
        "management_process_evidence": management_evidence,
        "board_decisions_2024": selected_decisions,
        "assurance_context": {
            "financed_emissions_2024_tco2e": financed_emissions_2024,
            "assurance_scope": assurance_scope,
            "financed_emissions_in_scope": financed_in_scope,
        },
        "strict_governance_evidence": {
            "formal_governance_mandate_available": formal_mandate_available,
            "formal_governance_mandate_instruction": (
                "Do not claim the ESG & Sustainability Committee has a formal climate mandate unless charter "
                "or terms-of-reference evidence is provided. If no formal instrument is available, say that "
                "available documentation evidences committee activity and meeting frequency but does not include "
                "the committee charter or terms of reference."
            ),
            "board_tradeoff_evidence_available": board_tradeoff_evidence_available,
            "tradeoff_decisions": tradeoff_decisions[:3],
            "board_tradeoff_instruction": (
                "Discuss board trade-offs only if explicit trade-off evidence exists. If not, state that the board "
                "decision evidence identifies climate-related decisions but does not describe specific trade-offs "
                "such as profitability, capital allocation, implementation cost, risk appetite or competing strategic priorities."
            ),
            "skills_adequacy_process_available": skills_adequacy_process_available,
            "skills_adequacy_instruction": (
                "Use the board climate expertise percentage and skills development programme as outcome/activity evidence. "
                "Do not invent a formal skills adequacy assessment process."
            ),
            "assurance_scope_limitation": assurance_scope_limitation,
        },
        "interpretation_notes": {
            "climate_on_board_agenda_pct": (
                "This figure represents the percentage of board meetings during the year where climate-related topics "
                "appeared on the agenda. It does not mean percentage of agenda time devoted to climate."
            ),
            "management_committee_names": (
                "Committee names are recorded by year only. The evidence does not prove that one committee evolved into, "
                "replaced, or was renamed as another. State the 2024 committee name and, if comparative names are used, "
                "present them neutrally."
            ),
            "major_transactions_climate_check": (
                "A true value indicates evidence of climate checks for major transactions; it does not prove a formal mandatory policy."
            ),
            "board_decision_traceability": (
                "Grouped decisions include internal refs for audit traceability. Do not print these refs in the final report."
            ),
            "avoid_duplication": (
                "Do not list the same board decisions twice. Board oversight should summarise decision governance; "
                "the detailed dated list belongs only in the Board and committee decisions subsection."
            ),
            "carbon_credit_boundary": (
                "Carbon credit procurement may be described only as a specific 2024 budget approval decision unless recurring topic evidence is available."
            ),
            "alignment_boundary": (
                "TCFD/IFRS S2 alignment flags show source-data alignment status only. Do not claim governance oversight supports or ensures alignment unless an alignment control process is evidenced."
            ),
        },
    }


evidence = extract_governance_evidence(governance_payload)

print(f"Evidence extracted for: {evidence['bank']['name']}")
print(f"Governance payload risk register present: {evidence['payload_profile']['climate_risk_register_in_payload']}")
print(f"Board decisions selected: {len(evidence['board_decisions_2024'])}")
print(f"Trend years: {[t['year'] for t in evidence['governance_trend']]}")
print(f"Management evidence risk register available: {evidence['management_process_evidence'].get('risk_register_available')}")
print("Strict governance evidence flags:")
for k, v in evidence["strict_governance_evidence"].items():
    if isinstance(v, bool):
        print(f"- {k}: {v}")
print("Grouped decisions:")
for d in evidence["board_decisions_2024"]:
    print(f"- {d.get('primary_date')} | {', '.join(d.get('committees', []))} | {d.get('decision')}")

In [ ]:
# ── GOVERNANCE EVIDENCE AVAILABILITY + SAVING ────────────────
# The raw Governance payload remains the source of truth.
# The compact evidence is the agent-ready input used by Writer/Judge/Reviser.

def is_scope1_scope2_only(scope: str | None) -> bool:
    if not scope:
        return False
    s = str(scope).lower()
    s = re.sub(r"\s+", " ", s)
    has_scope1 = bool(re.search(r"\bscope\s*1\b|\bscope1\b", s))
    has_scope2 = bool(
        re.search(r"\bscope\s*2\b|\bscope2\b", s)
        or re.search(r"\bscope\s*1\s*(and|&)\s*2\b", s)
        or re.search(r"\bscope\s*1\s*(and|&)\s*scope\s*2\b", s)
    )
    has_scope3_or_financed = bool(
        re.search(r"\bscope\s*3\b|\bscope3\b|financed emissions|category 15|cat\.?\s*15", s)
    )
    return has_scope1 and has_scope2 and not has_scope3_or_financed

def build_governance_availability_profile(evidence: dict) -> dict:
    gov = evidence.get("governance_2024", {})
    trend = evidence.get("governance_trend", [])
    management = evidence.get("management_process_evidence", {})
    strict = evidence.get("strict_governance_evidence", {})
    decisions = evidence.get("board_decisions_2024", [])

    def present(value) -> bool:
        return value is not None and str(value).strip().lower() not in {
            "", "none", "null", "nan"
        }

    trend_metrics = {
        "esg_committee_meetings": [
            row.get("esg_committee_meetings") for row in trend
            if present(row.get("esg_committee_meetings"))
        ],
        "board_climate_expertise_pct": [
            row.get("board_climate_expertise_pct") for row in trend
            if present(row.get("board_climate_expertise_pct"))
        ],
        "ceo_esg_compensation_pct": [
            row.get("ceo_esg_compensation_pct") for row in trend
            if present(row.get("ceo_esg_compensation_pct"))
        ],
        "all_exec_climate_remuneration_pct": [
            row.get("all_exec_climate_remuneration_pct") for row in trend
            if present(row.get("all_exec_climate_remuneration_pct"))
        ],
        "climate_on_board_agenda_pct": [
            row.get("climate_on_board_agenda_pct") for row in trend
            if present(row.get("climate_on_board_agenda_pct"))
        ],
        "board_full_meeting_frequency": [
            row.get("board_full_meeting_frequency") for row in trend
            if present(row.get("board_full_meeting_frequency"))
        ],
    }

    assurance_scope = str(gov.get("assurance_scope") or "").lower()
    financed_emissions = evidence.get("assurance_context", {}).get(
        "financed_emissions_2024_tco2e"
    )

    risk_register_available = bool(management.get("risk_register_available"))
    governance_controls_available = bool(management.get("governance_controls_available"))

    return {
        "board_core_metrics_available": all(
            present(gov.get(field))
            for field in [
                "board_size",
                "independent_directors_pct",
                "esg_committee_meetings_per_year",
                "climate_risk_reporting_to_board",
                "climate_on_board_agenda_pct",
            ]
        ),
        "trend_metrics_available": {
            name: len(values) >= 2
            for name, values in trend_metrics.items()
        },
        "selected_decision_count": len(decisions),
        "board_decisions_available": len(decisions) > 0,
        "formal_governance_mandate_available": bool(
            strict.get("formal_governance_mandate_available")
        ),
        "board_tradeoff_evidence_available": bool(
            strict.get("board_tradeoff_evidence_available")
        ),
        "management_process_available": risk_register_available or governance_controls_available,
        "management_risk_register_available": risk_register_available,
        "management_governance_controls_available": governance_controls_available,
        "formal_escalation_thresholds_available": bool(
            management.get("formal_escalation_thresholds_available", False)
        ),
        "skills_outcome_metrics_available": present(
            gov.get("board_climate_expertise_pct")
        ),
        "skills_development_programme_available": bool(
            gov.get("skills_development_programme")
        ),
        "skills_adequacy_process_available": bool(
            strict.get("skills_adequacy_process_available")
        ),
        "remuneration_evidence_available": (
            present(gov.get("ceo_esg_compensation_pct"))
            and present(gov.get("all_exec_climate_remuneration_pct"))
        ),
        "assurance_evidence_available": all(
            present(gov.get(field))
            for field in [
                "external_assurance",
                "assurance_provider",
                "assurance_scope",
                "assurance_standard",
            ]
        ),
        "assurance_scope_is_scope1_scope2_only": is_scope1_scope2_only(gov.get("assurance_scope")),
        "financed_emissions_available_for_scope_context": present(
            financed_emissions
        ),
        "payload_boundary": {
            "governance_payload_has_climate_risk_register": evidence.get("payload_profile", {}).get("climate_risk_register_in_payload"),
            "governance_payload_tables": evidence.get("payload_profile", {}).get("top_level_keys", []),
        },
        "writer_policy": {
            "use_available_evidence": (
                "Use every material governance evidence item that is available and relevant."
            ),
            "handle_unavailable_evidence": (
                "When a material governance requirement is not supported by the available evidence, state the boundary once in the relevant subsection. Do not invent the missing process or control."
            ),
            "management_process_boundary": (
                "If climate_risk_register is unavailable, describe only the management committee, board reporting frequency, ERM integration flag and major-transaction climate check. If it is available, describe the risk-register process but do not invent formal escalation thresholds."
            ),
            "wording": (
                "Use 'available evidence', 'available documentation', or 'source data' in final disclosure; do not use the word 'payload'."
            ),
        },
    }


def add_governance_traceability(evidence: dict, source_payload_path: Path) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_governance_availability_profile(evidence)
    source_tables = [
        "bank",
        "governance",
        "board_minutes",
        "reporting_kpis",
    ]
    if evidence.get("payload_profile", {}).get("climate_risk_register_in_payload"):
        source_tables.append("climate_risk_register")

    evidence["source_traceability"] = {
        "source_payload_path": str(source_payload_path),
        "source_tables": source_tables,
        "selected_board_decision_refs": [
            item.get("internal_ref")
            for item in evidence.get("board_decisions_2024", [])
            if item.get("internal_ref")
        ],
        "management_risk_refs": [
            item.get("risk_id")
            for item in evidence.get("management_process_evidence", {}).get(
                "material_risk_examples", []
            )
            if item.get("risk_id")
        ],
    }
    return evidence


evidence = add_governance_traceability(evidence, GOVERNANCE_PAYLOAD_PATH)

raw_governance_payload = {
    key: governance_payload.get(key)
    for key in [
        "metadata",
        "bank",
        "governance",
        "board_minutes",
        "climate_risk_register",
        "reporting_kpis",
    ]
    if key in governance_payload
}

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

RAW_GOVERNANCE_PATH = output_dir / "payload_BANK01_governance_raw.json"
COMPACT_GOVERNANCE_PATH = output_dir / "compact_governance_evidence_BANK01.json"

with open(RAW_GOVERNANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_governance_payload, f, indent=2, ensure_ascii=False)

with open(COMPACT_GOVERNANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(evidence, f, indent=2, ensure_ascii=False)

print("Governance evidence prepared")
print(f"- Raw Governance payload: {RAW_GOVERNANCE_PATH}")
print(f"- Compact Governance evidence: {COMPACT_GOVERNANCE_PATH}")
print("- Availability profile:")
print(json.dumps(evidence["availability_profile"], indent=2, ensure_ascii=False))

In [ ]:
# ── STATE DEFINITION ─────────────────────────────────────────
class GovernanceState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

In [ ]:
# ── GOVERNANCE REQUIREMENTS ─────────────────────────────────
# IFRS S1/S2 references are used internally only. Final text must not show paragraph tags.

IFRS_GOVERNANCE_REQUIREMENTS = """
STRICT GOVERNANCE DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Final section title and headings must be exactly:
### Governance
#### Board oversight
#### Management responsibility
#### Climate skills and competencies
#### Remuneration and climate incentives
#### Board and committee decisions during 2024
#### External assurance and controls

Internal IFRS S2 Governance coverage map:
1. Governance mandate / terms of reference / role descriptions:
   - If formal mandate evidence is available, describe it.
   - If unavailable, state that available documentation evidences activity and meeting frequency but does not include the charter, terms of reference or formal mandate.
2. Skills and competencies:
   - Use board climate expertise percentage and skills development programme.
   - If no formal skills adequacy assessment is available, state this boundary.
3. How often the governance body is informed:
   - Use climate risk reporting frequency to the Board.
   - Use climate_on_board_agenda_pct and board meeting frequency.
4. Trade-offs:
   - Use only explicit trade-off evidence.
   - If unavailable, state that documented decisions do not describe specific trade-offs.
5. Target-setting, progress monitoring and remuneration:
   - Use board/committee decisions related to transition plan, net-zero interim target revision, scenario analysis methodology and ESG report approval.
   - Use CEO and all-executive remuneration percentages.
6. Management responsibility:
   - Identify the management committee responsible for climate-related risks and opportunities.
7. Management controls and procedures:
   - If risk register evidence is available, describe the process: risk identification, categorisation, monitoring frequency, scenario links, mitigation actions and ERM integration.
   - If unavailable, limit the description to management committee, board reporting, ERM integration and major-transaction climate checks.
   - In both cases, do not invent formal escalation thresholds.

Data-aware coverage requirements:
1. Use the Governance evidence as the source of governance facts.
2. If climate_risk_register is present, Management responsibility must describe the evidenced risk-register process.
3. If climate_risk_register is absent, Management responsibility must be limited to management committee name, climate risk reporting frequency, ERM integration flag and major-transaction climate check flag.
4. Do not invent formal committee charters, terms of reference, formal governance mandates, formal escalation thresholds, board trade-off analysis or formal skills adequacy assessment.
5. Use year-on-year trends for available metrics: ESG committee meetings 5→6→7; board climate expertise 33.5%→35.5%→37.5%; CEO ESG-linked compensation 6.3%→7.8%→9.3%; all-executive climate-linked remuneration 8.7%→15.3%→15.1%; climate agenda frequency 69.8%→73.1%→72.6%; full Board meetings 4→7→8.
6. Use grouped 2024 board decisions. Put the detailed dated list only in the dedicated decisions subsection.
7. Carbon credit procurement may be described only as a specific 2024 budget approval decision unless recurring topic evidence is available.
8. Describe assurance only for the stated scope. If assurance scope is Scope 1 and 2 emissions, state that financed emissions / Scope 3 are outside the stated assurance scope when financed emissions context is available.
9. Alignment flags may be disclosed only as source-data flags. Do not claim governance oversight supports or ensures TCFD/IFRS S2 alignment unless an alignment control process is evidenced.
10. Do not include visible IFRS paragraph references, meeting IDs, internal refs or bracketed IFRS tags in the final output.
11. Use "available evidence", "available documentation" or "source data" in limitation wording. Do not use the technical word "payload" in the final report.
""".strip()

print("Governance requirements ready")


In [ ]:
# ── GOVERNANCE WRITER PROMPT ─────────────────────────────────
WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Governance section of an IFRS S1/S2-aligned climate disclosure report for a commercial bank.

Write in a formal, third-person, publication-ready style.

Evidence rules:
- Use only the compact Governance evidence supplied by the user prompt.
- Do not invent missing governance policies, committee charters, trade-offs, escalation thresholds, risk-register workflows, assurance coverage or skills assessment processes.
- When evidence is missing, state the limitation once in report-style language using "available evidence", "available documentation" or "source data".
- Do not use the word "payload" in the final section.
- Do not include IFRS paragraph references, meeting IDs, internal references or bracketed evidence tags.
- Do not use markdown tables.

Wording rules:
- Avoid strong interpretive verbs such as "demonstrates", "ensures", "supports", "confirms" or "drives" unless the evidence directly proves the mechanism.
- Prefer "indicates", "is evidenced by", "is reflected in" or "the available evidence shows".
- Do not claim that governance oversight supports or ensures TCFD/IFRS S2 alignment; state only that source data flags alignment if relevant.

Output rules:
- Return only the complete Governance section.
- Keep exactly the six required subsections.
""".strip()


def build_writer_prompt(evidence: dict, judge_feedback: str = None) -> str:
    profile = evidence.get("availability_profile", {})
    gov = evidence.get("governance_2024", {})
    management = evidence.get("management_process_evidence", {})
    strict = evidence.get("strict_governance_evidence", {})

    instructions = []

    instructions.append(
        "- Use board size, independent-director percentage, full-board meeting count, climate agenda percentage, committee meeting count and board reporting frequency."
    )
    instructions.append(
        "- Include the available three-year trend narrative for ESG committee meetings, board climate expertise, CEO ESG-linked pay, all-executive climate-linked pay, climate agenda frequency and full Board meeting frequency."
    )

    if profile.get("formal_governance_mandate_available"):
        instructions.append("- Describe the formal governance mandate using the supplied direct evidence.")
    else:
        instructions.append(
            "- Committee activity is evidenced, but no committee charter/terms of reference/formal mandate is available. State this boundary once in Board oversight."
        )

    if profile.get("board_tradeoff_evidence_available"):
        instructions.append("- Describe only the documented board trade-offs included in evidence.")
    else:
        instructions.append(
            "- Board decisions are evidenced, but specific trade-offs are not documented. State this boundary once without inventing trade-offs."
        )

    if profile.get("management_risk_register_available"):
        instructions.append(
            "- Management responsibility must describe the available risk-register process: eight 2024 risks, risk categories, risk ratings, time horizons, monitoring frequencies, scenario links, mitigation actions and ERM integration count."
        )
    elif profile.get("management_governance_controls_available"):
        instructions.append(
            "- Management responsibility must NOT describe a risk-register workflow. Use only the Climate Risk Management Committee, semi-annual board reporting, ERM integration flag and major-transaction climate check."
        )
    else:
        instructions.append(
            "- Management process evidence is not available. State the boundary without inventing a process."
        )

    if profile.get("formal_escalation_thresholds_available"):
        instructions.append("- Describe formal escalation thresholds/routes exactly as evidenced.")
    else:
        instructions.append("- Formal escalation thresholds are not evidenced. Use only boundary wording such as: available documentation does not evidence formal escalation thresholds or trigger-based escalation mechanics.")

    if profile.get("skills_adequacy_process_available"):
        instructions.append("- Describe the formal board skills adequacy assessment process from evidence.")
    else:
        instructions.append(
            "- Use board climate expertise percentage and skills development programme only. State that no formal board skills adequacy assessment process is documented. Do not invent detailed training topics."
        )

    if profile.get("remuneration_evidence_available"):
        instructions.append(
            "- Use both CEO ESG-linked remuneration percentage and all-executive climate-linked remuneration percentage, including available comparative trend values."
        )

    if profile.get("assurance_evidence_available"):
        instructions.append(
            "- Describe assurance using exact provider, level, standard and scope."
        )
        if profile.get("assurance_scope_is_scope1_scope2_only"):
            instructions.append(
                "- Clarify that assurance covers Scope 1 and Scope 2 emissions only, and financed emissions / Scope 3 are outside the stated assurance scope."
            )

    feedback_block = ""
    if judge_feedback:
        feedback_block = f"""
JUDGE FEEDBACK TO ADDRESS:
{judge_feedback}

REVISION RULES:
- Fix the judge's valid issues using available evidence.
- Preserve correct evidence-boundary statements.
- Never invent unavailable information.
""".strip()

    return f"""
{IFRS_GOVERNANCE_REQUIREMENTS}

BANK:
{evidence['bank']['name']} ({evidence['bank']['country']})
REPORTING YEAR: {evidence['reporting_year']}
COMPARATIVE YEARS: {evidence['comparative_years']}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DATA-AWARE WRITING INSTRUCTIONS:
{chr(10).join(instructions)}

COMPACT GOVERNANCE EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

IMPORTANT DATA BOUNDARIES:
- Governance evidence tables available: {evidence.get('payload_profile', {}).get('top_level_keys')}
- Climate risk register present in Governance evidence: {evidence.get('payload_profile', {}).get('climate_risk_register_in_payload')}
- Management process instruction: {management.get('process_flow_instruction')}
- Formal mandate instruction: {strict.get('formal_governance_mandate_instruction')}
- Board trade-off instruction: {strict.get('board_tradeoff_instruction')}
- Skills instruction: {strict.get('skills_adequacy_instruction')}
- Assurance instruction: {strict.get('assurance_scope_limitation', {}).get('instruction')}

GENERAL WRITING RULES:
- Do not duplicate the detailed board-decision list in Board oversight.
- Use dates and decision descriptions only in the dedicated decisions subsection.
- Interpret climate_on_board_agenda_pct as the percentage of board meetings where climate appeared on the agenda.
- A true major_transactions_climate_check flag indicates evidence of climate checks; it does not prove a formal mandatory policy.
- Do not infer that committee names from 2022, 2023 and 2024 represent the same renamed committee.
- Do not describe carbon credit procurement as recurring governance unless topic evidence explicitly supports that. If it appears only as a decision title, mention it only in the Board and committee decisions subsection.
- When alignment flags such as TCFD-aligned or IFRS S2-aligned are present, state only that source data flags the disclosure as aligned; do not claim governance oversight supports or ensures alignment.

{feedback_block}

Write the complete Governance section now.
Return only the final Governance section.
""".strip()

In [ ]:
# ── GOVERNANCE JUDGE PROMPT ─────────────────────────────────
JUDGE_SYSTEM = """
You are a strict sustainability-reporting judge for an IFRS S1/S2-aligned bank climate disclosure.

You evaluate whether the Governance section is:
- supported by the compact evidence;
- aligned with the Governance disclosure checklist;
- transparent about missing evidence;
- free from hallucinated policies, processes, trade-offs, assurance coverage and unsupported governance claims.

Return valid JSON only.
All checklist values must be JSON booleans true or false, not strings such as "true" or "false".
""".strip()


def build_judge_prompt(draft: str, evidence: dict, deterministic_checks: dict | None = None) -> str:
    profile = evidence.get("availability_profile", {})
    deterministic_checks = deterministic_checks or {}

    return f"""
Evaluate the Governance draft against the compact evidence, availability profile and deterministic pre-checks.

DRAFT:
{draft}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic_checks, indent=2, ensure_ascii=False)}

COMPACT GOVERNANCE EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
1. Penalise any claim that contradicts evidence or invents unavailable governance information.
2. Penalise omission when material evidence is available but not used.
3. Do not demand unavailable evidence. Correctly disclosed evidence boundaries are acceptable, but lower completeness.
4. Distinguish committee activity evidence from formal charter/mandate evidence.
5. Distinguish governance-level management controls from risk-register process evidence.
6. If climate_risk_register is available, the draft should describe the management risk-register process at a reasonable summary level: risk identification/register, risk categories, time horizons, ratings, monitoring frequencies, scenario links, mitigation actions and ERM integration.
7. If climate_risk_register is absent, the draft must not describe risk counts, risk categories, scenario links, monitoring frequencies by risk, mitigation actions or a risk-register workflow.
8. If formal mandate, board trade-offs, formal escalation thresholds or skills adequacy assessment are unavailable, the draft must not invent them.
9. Verify exact figures and trends: 10 board members; 68.5% independence; 72.6% 2024 climate agenda frequency; ESG Committee meetings 5→6→7; Board climate expertise 33.5%→35.5%→37.5%; CEO ESG remuneration 6.3%→7.8%→9.3%; all-executive climate remuneration 8.7%→15.3%→15.1%.
10. Verify that assurance is limited to Scope 1 and Scope 2 emissions and does not imply financed emissions assurance.
11. Verify no visible IFRS paragraph references, meeting IDs or internal refs appear in the final section.
12. Verify decisions are not duplicated between Board oversight and the dedicated decisions subsection.
13. Do not penalise accurate limitation statements such as "available documentation does not evidence formal escalation thresholds".
14. Alignment flags should be described only as source-data flags, not as proof that governance oversight ensures alignment.

SCORING:
- 9–10: Directly evidenced, materially complete, no unsupported claims and no material evidence boundary.
- 8: Strong, with only one material boundary correctly disclosed.
- 7: Usable, with multiple correctly disclosed boundaries.
- 6: Revision required because available evidence is omitted, or unsupported wording remains.
- 5 or below: Major evidence failure, hallucination, contradiction or missing core subsection.

Return valid JSON only. All checklist values must be booleans true/false, not strings.
Keep arrays concise:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "board_metrics_used_correctly": <true/false>,
    "trend_metrics_used_correctly": <true/false>,
    "board_decisions_used_correctly": <true/false>,
    "formal_mandate_handled_according_to_availability": <true/false>,
    "board_tradeoffs_handled_according_to_availability": <true/false>,
    "management_evidence_handled_according_to_availability": <true/false>,
    "risk_register_used_if_available_or_not_invented_if_absent": <true/false>,
    "escalation_handled_according_to_availability": <true/false>,
    "skills_handled_according_to_availability": <true/false>,
    "remuneration_evidence_used_correctly": <true/false>,
    "assurance_scope_used_correctly": <true/false>,
    "no_unsupported_committee_evolution": <true/false>,
    "no_duplicate_decisions": <true/false>,
    "no_visible_ifrs_refs_or_internal_refs": <true/false>,
    "no_unsupported_strong_claims": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted>],
  "unsupported_claims": [<specific unsupported claims>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()

In [ ]:
# ── GOVERNANCE EVALUATION MODE ───────────────────────────────
# Data-aware deterministic pre-checks are now active.
# They do not directly approve/reject the section or cap the score.
# Instead, their findings are passed to the GPT-5.2 Judge so the Judge can evaluate
# the draft against the actual available Governance data.

print("Governance evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks")

In [ ]:
# ── GOVERNANCE LANGGRAPH NODES ───────────────────────────────

def normalize_json_booleans(obj):
    """Normalize LLM JSON outputs where booleans may be returned as strings."""
    if isinstance(obj, dict):
        return {k: normalize_json_booleans(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [normalize_json_booleans(v) for v in obj]
    if isinstance(obj, str):
        if obj.strip().lower() == "true":
            return True
        if obj.strip().lower() == "false":
            return False
    return obj

def _contains_any(text: str, phrases: list[str]) -> bool:
    text_l = text.lower()
    return any(p.lower() in text_l for p in phrases)


def run_governance_deterministic_checks(draft: str, evidence: dict) -> dict:
    """Data-aware deterministic checks passed to the Judge as evidence-aware signals."""
    text = draft or ""
    text_l = text.lower()
    profile = evidence.get("availability_profile", {})
    gov = evidence.get("governance_2024", {})

    required_headings = [
        "#### Board oversight",
        "#### Management responsibility",
        "#### Climate skills and competencies",
        "#### Remuneration and climate incentives",
        "#### Board and committee decisions during 2024",
        "#### External assurance and controls",
    ]

    missing_headings = [h for h in required_headings if h not in text]

    warnings = []
    failures = []

    if missing_headings:
        failures.append({"check": "missing_required_headings", "details": missing_headings})

    if re.search(r"IFRS\s*S?[12]?\s*§|§\s*\d|\[IFRS", text):
        failures.append({"check": "visible_ifrs_references", "details": "Visible IFRS paragraph references/tags found."})

    if "[REF:" in text or re.search(r"MTG-BANK\d+", text):
        failures.append({"check": "internal_refs_visible", "details": "Internal meeting references should not appear in final text."})

    required_numbers = {
        "missing_climate_agenda_pct": ("72.6", "72.6% climate agenda frequency is available and must be used."),
        "missing_board_climate_expertise_pct": ("37.5", "37.5% board climate expertise is available and must be used."),
        "missing_ceo_esg_remuneration_pct": ("9.3", "9.3% CEO ESG-linked remuneration is available and must be used."),
        "missing_all_exec_remuneration_pct": ("15.1", "15.1% all-executive climate-linked remuneration is available and must be used."),
    }
    for check_name, (num, detail) in required_numbers.items():
        if num not in text:
            failures.append({"check": check_name, "details": detail})

    required_trend_patterns = {
        "esg_committee_trend": ["5", "6", "7"],
        "board_expertise_trend": ["33.5", "35.5", "37.5"],
        "ceo_remuneration_trend": ["6.3", "7.8", "9.3"],
        "all_exec_remuneration_trend": ["8.7", "15.3", "15.1"],
        "climate_agenda_trend": ["69.8", "73.1", "72.6"],
        "board_meeting_trend": ["4", "7", "8"],
    }
    for check_name, values in required_trend_patterns.items():
        if not all(v in text for v in values):
            failures.append({"check": f"missing_{check_name}", "details": f"Expected trend values missing: {values}"})

    if not profile.get("formal_governance_mandate_available"):
        if _contains_any(text, ["formal mandate", "committee charter", "terms of reference"]) and not _contains_any(text, ["does not include", "not documented", "not available", "available documentation does not"]):
            failures.append({"check": "formal_mandate_overclaimed", "details": "Formal mandate/charter language used without a limitation."})

    if not profile.get("board_tradeoff_evidence_available"):
        if _contains_any(text, ["trade-off", "tradeoff", "capital allocation", "profitability", "risk appetite"]) and not _contains_any(text, ["not describe", "not documented", "not available", "does not provide"]):
            failures.append({"check": "board_tradeoffs_overclaimed", "details": "Trade-off language used without direct evidence or limitation."})

    if profile.get("management_risk_register_available"):
        required_terms = ["risk register", "risk categor", "monitoring", "erm"]
        if not all(term in text_l for term in required_terms):
            failures.append({"check": "risk_register_process_omitted", "details": "Risk-register evidence is available and should be summarised in Management responsibility."})
    else:
        if _contains_any(text, ["risk register", "risk categories", "scenario links", "mitigation actions", "monitoring frequencies"]) and not _contains_any(text, ["does not contain", "not available", "not documented", "available evidence does not"]):
            failures.append({"check": "risk_register_process_invented", "details": "Risk-register process language appears although risk register is absent from Governance evidence."})

    if not profile.get("formal_escalation_thresholds_available"):
        escalation_terms = [
            "escalation threshold", "formal escalation", "escalation trigger",
            "trigger-based escalation", "defined escalation pathway", "escalation route"
        ]
        allowed_boundary = _contains_any(text, [
            "not documented", "not available", "does not specify", "does not evidence",
            "not evidenced", "no formal escalation",
            "available documentation does not evidence",
            "no such structures are therefore described"
        ])
        if any(term in text_l for term in escalation_terms) and not allowed_boundary:
            failures.append({"check": "escalation_threshold_overclaimed", "details": "Formal escalation language appears without evidence."})

    if not profile.get("skills_adequacy_process_available"):
        if _contains_any(text, ["skills adequacy assessment", "skills matrix", "skills gap analysis", "formal skills assessment"]) and not _contains_any(text, ["not documented", "not available", "does not describe"]):
            failures.append({"check": "skills_process_overclaimed", "details": "Formal skills adequacy process appears without evidence."})

    if profile.get("assurance_scope_is_scope1_scope2_only"):
        if "assurance" in text_l and "financed emissions" not in text_l:
            warnings.append({"check": "financed_emissions_scope_boundary_missing", "details": "Assurance section may not clearly state financed emissions are outside assurance scope."})
        if _contains_any(text, ["financed emissions are assured", "scope 3 emissions are assured", "assurance over financed emissions"]):
            failures.append({"check": "assurance_scope_overclaimed", "details": "Draft implies financed emissions / Scope 3 are assured."})

    forbidden_strong_phrases = [
        "governance oversight supports the bank's tcfd",
        "governance oversight supports the bank’s tcfd",
        "governance oversight ensures",
        "ensures alignment",
        "supports alignment",
        "fully aligned",
    ]
    found_strong = [p for p in forbidden_strong_phrases if p in text_l]
    if found_strong:
        failures.append({"check": "unsupported_alignment_or_strong_claim", "details": found_strong})

    return {
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


def writer_node(state: GovernanceState) -> GovernanceState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_writer_prompt(state["evidence"], judge_feedback=feedback)

    draft = call_writer_llm(
        system_prompt=WRITER_SYSTEM,
        user_prompt=prompt,
    )

    print(f"\n{'='*50}")
    print(f"WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {
        **state,
        "draft": draft.strip(),
        "status": "judging"
    }


def judge_node(state: GovernanceState) -> GovernanceState:
    draft = state["draft"]
    deterministic_checks = run_governance_deterministic_checks(draft, state["evidence"])

    judge_prompt = build_judge_prompt(
        draft,
        state["evidence"],
        deterministic_checks=deterministic_checks,
    )
    judge_result = call_judge_llm_json(
        system_prompt=JUDGE_SYSTEM,
        user_prompt=judge_prompt,
    )
    judge_result = normalize_json_booleans(judge_result)

    judge_result.setdefault("approved", False)
    judge_result.setdefault(
        "approval_status",
        "approved" if judge_result.get("approved") else "revision_required",
    )
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})
    judge_result["deterministic_prechecks"] = deterministic_checks

    print("\nGOVERNANCE JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))

    return {
        **state,
        "judge_result": judge_result,
        "status": "judging",
    }


GOVERNANCE_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Governance section using only:
- the supplied compact Governance evidence;
- the availability profile;
- the data-aware deterministic pre-checks; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- If risk-register evidence is unavailable, do not describe a risk-register workflow.
- Keep the exact six-subsection structure.
- Do not add visible IFRS paragraph references, meeting IDs or internal refs.
- Return only the complete revised Governance section.
""".strip()


def reviser_node(state: GovernanceState) -> GovernanceState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}

    judge = state.get("judge_result", {})
    issues = judge.get("required_fixes", [])
    checklist = judge.get("checklist", {})
    deterministic = judge.get("deterministic_prechecks", {})
    false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]

    revision_prompt = f"""
STRICT GOVERNANCE REQUIREMENTS:
{IFRS_GOVERNANCE_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic, indent=2, ensure_ascii=False)}

COMPACT GOVERNANCE EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- When evidence is unavailable, preserve or improve the accurate boundary statement.
- Never invent a missing process, policy, threshold, trade-off, risk-register workflow, or assurance scope.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete revised Governance section.
""".strip()

    revised_draft = call_reviser_llm(
        system_prompt=GOVERNANCE_REVISER_SYSTEM,
        user_prompt=revision_prompt,
    )

    new_revision_count = state["revision_count"] + 1
    print(f"\nGovernance revised with GPT-4.1 | revision {new_revision_count}")

    return {
        **state,
        "draft": revised_draft.strip(),
        "revision_count": new_revision_count,
        "status": "judging",
    }


def finalize_node(state: GovernanceState) -> GovernanceState:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)

    print(f"\n{'='*50}")
    print(f"FINALIZED")
    print(f"  Status: {'APPROVED' if approved else 'MAX REVISIONS REACHED'}")
    print(f"  Final score: {judge.get('overall_score')}/10")
    print(f"  Revisions: {state['revision_count']}")
    print(f"{'='*50}")

    return {
        **state,
        "final_section": state["draft"],
        "status": "approved" if approved else "failed"
    }


def route_after_judge(state: GovernanceState) -> str:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)
    revision_count = state.get("revision_count", 0)
    max_revisions = state.get("max_revisions", 2)

    if approved:
        return "finalize"
    if revision_count >= max_revisions:
        return "finalize"
    return "revise"

In [ ]:
# ── BUILD AND COMPILE GRAPH ───────────────────────────────────
builder = StateGraph(GovernanceState)

builder.add_node("writer",   writer_node)
builder.add_node("judge",    judge_node)
builder.add_node("reviser",  reviser_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START,      "writer")
builder.add_edge("writer",   "judge")
builder.add_edge("reviser",  "judge")
builder.add_edge("finalize", END)

builder.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "revise":   "reviser",
        "finalize": "finalize",
    }
)

graph = builder.compile()
print("Graph compiled")

In [ ]:
# ── RUN ──────────────────────────────────────────────────────
initial_state: GovernanceState = {
    "bank_name":      bank_name,
    "evidence":       evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  2,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {}
}

print(f"Starting governance generation for: {bank_name}\n")
result = graph.invoke(initial_state)

In [ ]:
# ── OUTPUT ───────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL JUDGE RESULT")
print("="*60)
print(json.dumps(result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("GOVERNANCE SECTION")
print("="*60)
print(result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "governance_BANK01.md", "w", encoding="utf-8") as f:
    f.write(result["final_section"])

with open(output_dir / "governance_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "governance",
        "status":          result["status"],
        "approval_status": result["judge_result"].get("approval_status"),
        "raw_score":       result["judge_result"].get("raw_score_before_caps"),
        "final_score":     result["judge_result"].get("overall_score"),
        "score_cap_reason": result["judge_result"].get("score_cap_reason"),
        "revisions":       result["revision_count"],
        "approved":        result["judge_result"].get("approved"),
        "checklist":       result["judge_result"].get("checklist"),
        "issues":         result["judge_result"].get("main_issues"),
        "available_evidence_omitted": result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims": result["judge_result"].get("unsupported_claims"),
        "correctly_disclosed_evidence_boundaries": result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_governance_payload_path": str(RAW_GOVERNANCE_PATH),
        "compact_governance_evidence_path": str(COMPACT_GOVERNANCE_PATH),
    }, f, indent=2, ensure_ascii=False)

print(f"\nSaved to outputs/governance_BANK01.md")
print(f"Raw Governance payload saved to: {RAW_GOVERNANCE_PATH}")
print(f"Compact Governance evidence saved to: {COMPACT_GOVERNANCE_PATH}")


In [ ]:

# ============================================================
# STRATEGY SECTION GENERATOR
# IFRS S1/S2 strategy | uses same Azure REST helper functions
# ============================================================
# This section is added after Governance and reuses:
# - payload
# - bank_name
# - call_writer_llm()
# - call_judge_llm_json()
# - call_reviser_llm()
# - _is_present(), _safe_int(), _normalise_text()


def _safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        # handle NaN from JSON payloads
        if isinstance(value, float) and value != value:
            return default
        return float(value)
    except Exception:
        return default


def _json_clean(obj):
    """Make dictionaries/lists safe for JSON prompt dumps by replacing NaN with None."""
    if isinstance(obj, dict):
        return {k: _json_clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_json_clean(v) for v in obj]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def _pick_latest(records: list[dict], year: int = 2024) -> dict:
    for r in records:
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year:
            return r
    return records[-1] if records else {}


print("Strategy helper functions ready")


In [ ]:
# ── FAST STRATEGY EVIDENCE EXTRACTOR ────────────────────────
# This version summarizes the Strategy payload before sending it to the LLM.
# It reflects the actual section-specific strategy payload:
# - climate_scenarios are present;
# - climate_risk_register is present;
# - value_chain_map is present;
# - climate_opportunities are present;
# - full top-level targets may be absent, but reporting_kpis.target_summary is available.

def _present(v) -> bool:
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip() != "" and v.strip().lower() not in {"nan", "none", "null"}
    try:
        return not (isinstance(v, float) and v != v)
    except Exception:
        return True


def _num(v):
    try:
        if not _present(v):
            return None
        return float(v)
    except Exception:
        return None


def _compact_clean(obj):
    if isinstance(obj, dict):
        cleaned = {k: _compact_clean(v) for k, v in obj.items() if _present(v)}
        return {k: v for k, v in cleaned.items() if v not in ({}, [], None)}
    if isinstance(obj, list):
        cleaned = [_compact_clean(x) for x in obj if _present(x)]
        return [x for x in cleaned if x not in ({}, [], None)]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def _top_n(rows: list[dict], key: str, n: int = 5) -> list[dict]:
    return sorted(rows, key=lambda r: float(r.get(key) or 0), reverse=True)[:n]


def summarize_strategy_evidence(payload: dict) -> dict:
    metadata = payload.get("metadata", {})
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})

    scenarios = [s for s in payload.get("climate_scenarios", []) if isinstance(s, dict)]
    risks_all = [r for r in payload.get("climate_risk_register", []) if isinstance(r, dict)]
    risks_2024 = [r for r in risks_all if r.get("reporting_year") == 2024]
    value_chain = [v for v in payload.get("value_chain_map", []) if isinstance(v, dict)]
    opportunities = [o for o in payload.get("climate_opportunities", []) if isinstance(o, dict) and o.get("reporting_year") == 2024]

    full_targets = [t for t in payload.get("targets", []) if isinstance(t, dict)]
    target_summary = reporting_kpis.get("target_summary", [])
    targets = full_targets if full_targets else target_summary

    physical_risks = [r for r in risks_2024 if str(r.get("risk_category", "")).startswith("physical")]
    transition_risks = [r for r in risks_2024 if str(r.get("risk_category", "")).startswith("transition")]

    risk_summary = {
        "risk_count_2024": len(risks_2024),
        "physical_risk_count": len(physical_risks),
        "transition_risk_count": len(transition_risks),
        "time_horizons": sorted({r.get("time_horizon") for r in risks_2024 if _present(r.get("time_horizon"))}),
        "risk_categories": sorted({r.get("risk_category") for r in risks_2024 if _present(r.get("risk_category"))}),
        "risk_ratings": sorted({r.get("risk_rating") for r in risks_2024 if _present(r.get("risk_rating"))}),
        "top_risks_by_financial_impact": [
            {
                "risk_id": r.get("risk_id"),
                "risk_name": r.get("risk_name"),
                "risk_category": r.get("risk_category"),
                "risk_rating": r.get("risk_rating"),
                "time_horizon": r.get("time_horizon"),
                "financial_impact_meur": r.get("financial_impact_meur"),
                "mitigation_actions": r.get("mitigation_actions"),
                "scenario_analysis_link": r.get("scenario_analysis_link"),
            }
            for r in _top_n(risks_2024, "financial_impact_meur", 6)
        ],
    }

    scenario_types = sorted({s.get("scenario_type") for s in scenarios if _present(s.get("scenario_type"))})
    scenario_names = sorted({s.get("scenario_name") for s in scenarios if _present(s.get("scenario_name"))})
    frameworks = sorted({s.get("framework") for s in scenarios if _present(s.get("framework"))})
    horizon_years = sorted({s.get("horizon_year") for s in scenarios if _present(s.get("horizon_year"))})

    def max_record(field: str):
        rows = [s for s in scenarios if _present(s.get(field))]
        if not rows:
            return None
        s = max(rows, key=lambda x: float(x.get(field) or 0))
        return {
            "scenario_id": s.get("scenario_id"),
            "scenario_name": s.get("scenario_name"),
            "scenario_type": s.get("scenario_type"),
            "horizon": s.get("horizon"),
            "horizon_year": s.get("horizon_year"),
            field: s.get(field),
        }

    resilience_by_type = {}
    methodology_by_type = {}
    for s in scenarios:
        stype = s.get("scenario_type")
        if not _present(stype):
            continue
        if _present(s.get("resilience_assessment")):
            resilience_by_type.setdefault(stype, s.get("resilience_assessment"))
        if _present(s.get("methodology_notes")):
            methodology_by_type.setdefault(stype, s.get("methodology_notes"))

    scenario_summary = {
        "available": bool(scenarios),
        "scenario_count": len(scenarios),
        "frameworks": frameworks,
        "scenario_types": scenario_types,
        "scenario_names": scenario_names,
        "horizon_years": horizon_years,
        "methodology_summary_by_scenario_type": methodology_by_type,
        "resilience_assessment_by_scenario_type": resilience_by_type,
        "max_physical_risk_loss_pct_capital": max_record("physical_risk_loss_pct_capital"),
        "max_transition_risk_loss_pct_capital": max_record("transition_risk_loss_pct_capital"),
        "max_stranded_assets_estimate_meur": max_record("stranded_assets_estimate_meur"),
        "max_revenue_at_risk_meur": max_record("revenue_at_risk_meur"),
        "key_assumption_ranges": {
            "carbon_price_eur_per_tco2e": {
                "min": min([s.get("carbon_price_assumption_eur_per_tco2e") for s in scenarios if _present(s.get("carbon_price_assumption_eur_per_tco2e"))], default=None),
                "max": max([s.get("carbon_price_assumption_eur_per_tco2e") for s in scenarios if _present(s.get("carbon_price_assumption_eur_per_tco2e"))], default=None),
            },
            "technology_readiness": sorted({s.get("technology_readiness") for s in scenarios if _present(s.get("technology_readiness"))}),
            "temperature_outcome_c": sorted({s.get("temperature_outcome_c") for s in scenarios if _present(s.get("temperature_outcome_c"))}),
        },
    }

    material_vc = [v for v in value_chain if v.get("materiality_flag") is True]
    quantified_vc = [v for v in material_vc if _present(v.get("financial_exposure_meur"))]

    value_chain_summary = {
        "available": bool(value_chain),
        "node_count": len(value_chain),
        "material_node_count": len(material_vc),
        "qualitative_nodes_without_financial_exposure": len([v for v in material_vc if not _present(v.get("financial_exposure_meur"))]),
        "node_types": sorted({v.get("node_type") for v in value_chain if _present(v.get("node_type"))}),
        "largest_quantified_nodes": [
            {
                "node_name": v.get("node_name"),
                "node_type": v.get("node_type"),
                "upstream_downstream": v.get("upstream_downstream"),
                "climate_exposure_type": v.get("climate_exposure_type"),
                "financial_exposure_meur": v.get("financial_exposure_meur"),
                "climate_risk_description": v.get("climate_risk_description"),
            }
            for v in _top_n(quantified_vc, "financial_exposure_meur", 6)
        ],
        "own_operations_examples": [
            {
                "node_name": v.get("node_name"),
                "climate_exposure_type": v.get("climate_exposure_type"),
                "scope3_category": v.get("scope3_category"),
                "climate_risk_description": v.get("climate_risk_description"),
            }
            for v in material_vc
            if v.get("node_type") == "own_operations"
        ][:3],
    }

    opportunity_summary = {
        "available": bool(opportunities),
        "opportunity_count": len(opportunities),
        "total_estimated_revenue_impact_meur": round(sum(float(o.get("estimated_revenue_impact_meur") or 0) for o in opportunities), 2) if opportunities else None,
        "examples": [
            {
                "opportunity_id": o.get("opportunity_id"),
                "opportunity_type": o.get("opportunity_type"),
                "category": o.get("category"),
                "description": o.get("description"),
                "estimated_revenue_impact_meur": o.get("estimated_revenue_impact_meur"),
                "time_horizon": o.get("time_horizon"),
                "confidence_level": o.get("confidence_level"),
                "linked_risk_category": o.get("linked_risk_category"),
            }
            for o in opportunities
        ],
    }

    financial_2024 = next(
        (f for f in payload.get("financial_summary", []) if isinstance(f, dict) and f.get("reporting_year") == 2024),
        {},
    )

    portfolio_summary = {
        "total_assets_meur": bank.get("total_assets_meur") or financial_2024.get("total_assets_meur"),
        "total_loans_meur": bank.get("total_loans_meur") or financial_2024.get("total_loans_meur"),
        "green_loans_meur": financial_2024.get("green_loans_meur"),
        "green_loans_pct": financial_2024.get("green_loans_pct") or reporting_kpis.get("green_loans_pct_2024"),
        "financed_emissions_tco2e": reporting_kpis.get("financed_emissions_2024_tco2e"),
        "carbon_intensity_tco2e_per_meur": reporting_kpis.get("carbon_intensity_2024_tco2e_per_meur"),
        "climate_capex_meur": reporting_kpis.get("climate_capex_2024_meur") or financial_2024.get("climate_capex_meur"),
        "climate_opex_meur": reporting_kpis.get("climate_opex_2024_meur") or financial_2024.get("climate_opex_meur"),
        "high_carbon_sector_exposure_pct": reporting_kpis.get("high_carbon_sector_exposure_pct"),
        "fossil_fuel_exposure_pct": reporting_kpis.get("fossil_fuel_exposure_pct"),
        "high_carbon_sector_exposure_meur": reporting_kpis.get("high_carbon_sector_exposure_meur"),
        "fossil_fuel_exposure_meur": reporting_kpis.get("fossil_fuel_exposure_meur"),
    }

    compact = {
        "bank": {
            "bank_id": bank.get("bank_id"),
            "bank_name": bank.get("bank_name"),
            "country": bank.get("country"),
            "reporting_year": metadata.get("reporting_year", 2024),
            "reporting_currency": bank.get("reporting_currency"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "total_loans_meur": bank.get("total_loans_meur"),
        },
        "payload_profile": {
            "source_payload": "strategy",
            "top_level_keys": list(payload.keys()),
            "full_targets_table_available": bool(full_targets),
            "target_summary_available": bool(target_summary),
            "climate_scenarios_count": len(scenarios),
            "risk_register_count": len(risks_all),
            "value_chain_node_count": len(value_chain),
            "opportunity_count": len(opportunities),
        },
        "risk_summary": risk_summary,
        "scenario_summary": scenario_summary,
        "value_chain_summary": value_chain_summary,
        "opportunity_summary": opportunity_summary,
        "portfolio_summary": portfolio_summary,
        "targets": targets,
        "business_model_impacts": payload.get("business_model_impacts", []),
        "strategy_tradeoff_decisions": payload.get("strategy_tradeoff_decisions", []),
        "evidence_boundaries": {
            "scenario_outputs_are_modelled_estimates": True,
            "opportunity_impacts_are_estimates": True,
            "do_not_overclaim_resilience": True,
            "do_not_overclaim_paris_alignment": True,
            "business_model_detail_available": bool(payload.get("business_model_impacts")),
            "value_chain_detail_available": bool(value_chain_summary),
            "tradeoff_evidence_available": bool(payload.get("strategy_tradeoff_decisions")),
            "high_carbon_and_fossil_exposure_available": _present(portfolio_summary.get("high_carbon_sector_exposure_pct")) and _present(portfolio_summary.get("fossil_fuel_exposure_pct")),
            "full_targets_table_available": bool(full_targets),
            "target_summary_available": bool(target_summary),
            "target_instruction": (
                "Only reporting_kpis.target_summary is available; do not invent target baselines, validation bodies, milestones, gross/net status or planned credit details."
                if not full_targets and target_summary else
                "Full target records are available; use only fields explicitly provided."
            ),
        },
    }
    return _compact_clean(compact)


strategy_evidence = summarize_strategy_evidence(strategy_payload)

raw_size = len(json.dumps(strategy_payload, ensure_ascii=False))
compact_size = len(json.dumps(strategy_evidence, ensure_ascii=False))
print("Compact Strategy evidence ready")
print(f"Raw strategy payload size: {raw_size:,} chars")
print(f"Compact evidence size: {compact_size:,} chars")
print(f"Reduction: {round((1 - compact_size / max(raw_size, 1)) * 100, 1)}%")
print(json.dumps({
    "bank": strategy_evidence["bank"],
    "scenario_count": strategy_evidence["scenario_summary"].get("scenario_count"),
    "risk_count_2024": strategy_evidence["risk_summary"].get("risk_count_2024"),
    "value_chain_available": strategy_evidence["value_chain_summary"].get("available", True),
    "opportunities_available": strategy_evidence["opportunity_summary"].get("available", True),
    "full_targets_table_available": strategy_evidence["payload_profile"].get("full_targets_table_available"),
    "target_summary_available": strategy_evidence["payload_profile"].get("target_summary_available"),
    "business_model_impacts": len(strategy_evidence.get("business_model_impacts", [])),
    "strategy_tradeoffs": len(strategy_evidence.get("strategy_tradeoff_decisions", [])),
    "high_carbon_pct": strategy_evidence["portfolio_summary"].get("high_carbon_sector_exposure_pct"),
    "fossil_fuel_pct": strategy_evidence["portfolio_summary"].get("fossil_fuel_exposure_pct"),
}, indent=2, ensure_ascii=False))

In [ ]:
# ── STRATEGY EVIDENCE AVAILABILITY + SAVING ──────────────────
# The raw Strategy payload remains the source of truth.
# The compact Strategy evidence is the agent-ready input used by Writer/Judge/Reviser.

def build_strategy_availability_profile(evidence: dict) -> dict:
    risk = evidence.get("risk_summary", {})
    scenarios = evidence.get("scenario_summary", {})
    value_chain = evidence.get("value_chain_summary", {})
    opportunities = evidence.get("opportunity_summary", {})
    portfolio = evidence.get("portfolio_summary", {})
    boundaries = evidence.get("evidence_boundaries", {})
    targets = evidence.get("targets", [])
    business_model_impacts = evidence.get("business_model_impacts", [])
    tradeoffs = evidence.get("strategy_tradeoff_decisions", [])

    def present(value) -> bool:
        return value is not None and str(value).strip().lower() not in {
            "", "none", "null", "nan"
        }

    scenario_assumptions_available = any(
        present(scenarios.get(field))
        for field in [
            "frameworks",
            "scenario_types",
            "scenario_names",
            "horizon_years",
            "methodology_summary_by_scenario_type",
            "key_assumption_ranges",
        ]
    )

    quantified_value_chain_available = bool(value_chain.get("largest_quantified_nodes"))
    opportunity_estimates_available = any(
        present(item.get("estimated_revenue_impact_meur"))
        for item in opportunities.get("examples", [])
        if isinstance(item, dict)
    ) or present(opportunities.get("total_estimated_revenue_impact_meur"))

    return {
        "physical_risk_evidence_available": bool(risk.get("physical_risk_count")),
        "transition_risk_evidence_available": bool(risk.get("transition_risk_count")),
        "time_horizons_available": bool(risk.get("time_horizons"))
        or bool(scenarios.get("horizon_years")),
        "business_model_detail_available": bool(business_model_impacts),
        "value_chain_detail_available": bool(value_chain),
        "quantified_value_chain_exposure_available": quantified_value_chain_available,
        "strategy_tradeoff_evidence_available": bool(tradeoffs),
        "portfolio_financial_metrics_available": any(
            present(portfolio.get(field))
            for field in [
                "green_loans_meur",
                "green_loans_pct",
                "financed_emissions_tco2e",
                "carbon_intensity_tco2e_per_meur",
            ]
        ),
        "resource_allocation_evidence_available": (
            present(portfolio.get("climate_capex_meur"))
            or present(portfolio.get("climate_opex_meur"))
        ),
        "high_carbon_exposure_available": (
            present(portfolio.get("high_carbon_sector_exposure_pct"))
            or present(portfolio.get("high_carbon_sector_exposure_meur"))
        ),
        "fossil_fuel_exposure_available": (
            present(portfolio.get("fossil_fuel_exposure_pct"))
            or present(portfolio.get("fossil_fuel_exposure_meur"))
        ),
        "climate_opportunities_available": bool(opportunities.get("examples")),
        "quantified_opportunity_estimates_available": opportunity_estimates_available,
        "scenario_analysis_available": bool(scenarios.get("available")),
        "scenario_assumptions_available": scenario_assumptions_available,
        "scenario_financial_outputs_available": any(
            scenarios.get(field)
            for field in [
                "max_physical_risk_loss_pct_capital",
                "max_transition_risk_loss_pct_capital",
                "max_stranded_assets_estimate_meur",
                "max_revenue_at_risk_meur",
            ]
        ),
        "resilience_evidence_available": bool(
            scenarios.get("resilience_assessment_by_scenario_type")
        ),
        "full_targets_table_available": bool(
            evidence.get("payload_profile", {}).get("full_targets_table_available")
        ),
        "target_summary_available": bool(
            evidence.get("payload_profile", {}).get("target_summary_available")
        ),
        "targets_available": bool(targets),
        "scenario_outputs_are_modelled_estimates": bool(
            boundaries.get("scenario_outputs_are_modelled_estimates", True)
        ),
        "opportunity_impacts_are_estimates": bool(
            boundaries.get("opportunity_impacts_are_estimates", True)
        ),
        "writer_policy": {
            "use_available_evidence": (
                "Use every material Strategy evidence item that is available and relevant."
            ),
            "handle_unavailable_evidence": (
                "When a material Strategy disclosure is unsupported by available evidence, state the boundary once in the relevant subsection and do not invent it."
            ),
            "target_boundary": boundaries.get("target_instruction"),
            "scenario_wording": (
                "Describe scenario financial outputs as modelled estimates, not actual losses."
            ),
            "opportunity_wording": (
                "Describe opportunity revenue impacts as estimates with confidence level, not guaranteed future revenue."
            ),
            "resilience_wording": (
                "Confine resilience statements to the relevant scenario assumptions and boundaries."
            ),
            "final_language": (
                "Use 'available evidence', 'available documentation', or 'source data'; do not use the word 'payload' in final disclosure."
            ),
        },
    }


def add_strategy_traceability(evidence: dict, source_payload_path: Path) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_strategy_availability_profile(evidence)
    evidence["source_traceability"] = {
        "source_payload_path": str(source_payload_path),
        "source_tables": [
            "bank",
            "financial_summary",
            "climate_scenarios",
            "climate_risk_register",
            "value_chain_map",
            "climate_opportunities",
            "reporting_kpis",
        ],
        "top_risk_refs": [
            item.get("risk_id") or item.get("risk_name")
            for item in evidence.get("risk_summary", {}).get("top_risks_by_financial_impact", [])
            if item.get("risk_id") or item.get("risk_name")
        ],
        "scenario_refs": [
            item.get("scenario_id")
            for item in [
                evidence.get("scenario_summary", {}).get("max_physical_risk_loss_pct_capital"),
                evidence.get("scenario_summary", {}).get("max_transition_risk_loss_pct_capital"),
                evidence.get("scenario_summary", {}).get("max_stranded_assets_estimate_meur"),
                evidence.get("scenario_summary", {}).get("max_revenue_at_risk_meur"),
            ]
            if isinstance(item, dict) and item.get("scenario_id")
        ],
        "value_chain_refs": [
            item.get("node_name")
            for item in evidence.get("value_chain_summary", {}).get("largest_quantified_nodes", [])
            if item.get("node_name")
        ],
        "opportunity_refs": [
            item.get("opportunity_id") or item.get("opportunity_type")
            for item in evidence.get("opportunity_summary", {}).get("examples", [])
            if item.get("opportunity_id") or item.get("opportunity_type")
        ],
    }
    return evidence


strategy_evidence = add_strategy_traceability(
    strategy_evidence,
    STRATEGY_PAYLOAD_PATH,
)

raw_strategy_payload = {
    key: strategy_payload.get(key)
    for key in [
        "metadata",
        "bank",
        "financial_summary",
        "climate_scenarios",
        "climate_risk_register",
        "value_chain_map",
        "climate_opportunities",
        "targets",
        "reporting_kpis",
        "business_model_impacts",
        "strategy_tradeoff_decisions",
    ]
    if key in strategy_payload
}

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

RAW_STRATEGY_PATH = output_dir / "payload_BANK01_strategy_raw.json"
COMPACT_STRATEGY_PATH = output_dir / "compact_strategy_evidence_BANK01.json"

with open(RAW_STRATEGY_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_strategy_payload, f, indent=2, ensure_ascii=False)

with open(COMPACT_STRATEGY_PATH, "w", encoding="utf-8") as f:
    json.dump(strategy_evidence, f, indent=2, ensure_ascii=False)

print("Strategy evidence prepared")
print(f"- Raw Strategy payload: {RAW_STRATEGY_PATH}")
print(f"- Compact Strategy evidence: {COMPACT_STRATEGY_PATH}")
print("- Availability profile:")
print(json.dumps(strategy_evidence["availability_profile"], indent=2, ensure_ascii=False))

In [ ]:

# ── STRATEGY STATE DEFINITION ───────────────────────────────
class StrategyState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict


In [ ]:
# ── STRATEGY REQUIREMENTS ───────────────────────────────────
# IFRS references are used internally only. Final text must not show paragraph references.

STRATEGY_REQUIREMENTS = """
STRICT STRATEGY DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Final section title and headings must be exactly:
### Strategy
#### Climate-related risks and opportunities
#### Effects on business model and value chain
#### Effects on strategy and decision-making
#### Financial effects and resource allocation
#### Climate resilience and scenario analysis
#### Strategy limitations and evidence boundaries

Data-aware coverage requirements:
1. Use the Strategy payload as the source of Strategy facts.
2. Use risk register, climate scenarios, value-chain map, climate opportunities, financial summary and reporting KPIs when present.
3. Distinguish physical risks from transition risks.
4. Cover time horizons only where risk-register or scenario evidence supports them.
5. Use value-chain evidence when available. If some material nodes lack financial exposure, state that boundary.
6. Use climate-opportunity evidence when available. Opportunity revenue impacts must be described as estimates, not guaranteed revenue.
7. Use high-carbon and fossil-fuel exposure metrics when present.
8. Use climate capex and opex as resource-allocation evidence when present.
9. Use target evidence carefully: if only reporting_kpis.target_summary is available, use only target type, scope, status, target year, framework and progress; do not invent baselines, validation bodies, gross/net status, milestones, planned credits or full target mechanics unless full target records are present.

Scenario analysis must cover:
- scenario framework used;
- scenario names/types;
- time horizons and horizon years;
- carbon price assumptions;
- temperature outcomes;
- technology readiness assumptions;
- macroeconomic assumptions where available;
- energy assumptions such as renewable energy share where available;
- methodology notes;
- scope of analysis;
- financial outputs as modelled estimates only.

Climate resilience must be bounded and should cover only available adaptation-capacity evidence:
- financial resources: use climate capex, climate opex, capital buffer or resource-allocation evidence if available;
- portfolio flexibility: use sector exposure limits, green loan growth, client engagement, decarbonisation glide-path monitoring or collateral overlays if available;
- investment effects: use climate capex/opex and transition finance indicators if available;
- if these are not available, state the evidence boundary and do not invent operational adaptation capacity.

Transition plan disclosure:
- Use updated transition plan and net-zero interim target evidence only if evidenced.
- Use target frameworks and progress from target_summary or full targets.
- If key transition-plan assumptions or dependencies are not available, state this boundary.
- Do not invent dependencies, policy assumptions, customer behaviour assumptions or financing dependencies.

Other boundaries:
- Scenario financial outputs are modelled estimates, not actual losses.
- Resilience statements must remain scenario-specific and must not claim overall bank resilience.
- Do not claim overall Paris alignment. If target/scenario flags indicate alignment, use cautious wording and keep it tied to the specific target or scenario evidence.
- If business_model_impacts or strategy_tradeoff_decisions are absent, state the evidence boundary once; do not invent detailed business-model impacts or trade-off analysis.
- Do not include visible IFRS paragraph references or bracketed IFRS tags in the final output.
- Use "available evidence", "available documentation" or "source data" in limitation wording. Do not use the word "payload" in the final report.
""".strip()

print("Strategy requirements ready")


In [ ]:
# ── STRATEGY WRITER / JUDGE PROMPTS ─────────────────────────
STRATEGY_WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Strategy section of an IFRS S1/S2-aligned climate disclosure report for a commercial bank.

Write in a formal, third-person, publication-ready style.

Evidence rules:
- Use only the compact Strategy evidence supplied by the user prompt.
- Do not invent strategy decisions, business-model impacts, trade-offs, value-chain nodes, opportunities, target details, scenario assumptions, financial effects or resilience conclusions.
- If evidence is missing, state the limitation once in report-style language using "available evidence", "available documentation" or "source data".
- Do not use the word "payload" in the final section.
- Do not include IFRS paragraph references or bracketed evidence tags.
- Do not use markdown tables.

Output rules:
- Return only the complete Strategy section.
- Keep exactly the six required subsections.
""".strip()


def build_strategy_writer_prompt(
    evidence: dict,
    judge_feedback: str | None = None,
) -> str:
    profile = evidence.get("availability_profile", {})
    boundaries = evidence.get("evidence_boundaries", {})
    instructions = []

    if profile.get("physical_risk_evidence_available") and profile.get("transition_risk_evidence_available"):
        instructions.append("- Distinguish and describe both physical and transition risks using the supplied risk examples.")
    else:
        instructions.append("- Describe only the risk types evidenced; do not invent missing risk categories.")

    if profile.get("time_horizons_available"):
        instructions.append("- Use available short-, medium- and long-term horizons where supported by risk/scenario evidence.")

    if profile.get("business_model_detail_available"):
        instructions.append("- Explain supplied business-model impacts and transmission channels.")
    else:
        instructions.append("- Detailed business_model_impacts evidence is absent. Use lending, portfolio, value-chain and financial evidence, and state that detailed business-model impact evidence is limited.")

    if profile.get("value_chain_detail_available"):
        instructions.append("- Use the value-chain map, including own operations, suppliers and financing counterparties.")
        if profile.get("quantified_value_chain_exposure_available"):
            instructions.append("- Include quantified value-chain exposures and state that some material nodes are qualitative where financial exposure is missing.")
    else:
        instructions.append("- Value-chain evidence is unavailable. Do not invent value-chain nodes or exposures.")

    if profile.get("strategy_tradeoff_evidence_available"):
        instructions.append("- Describe only documented Strategy trade-offs provided in evidence.")
    else:
        instructions.append("- No strategy_tradeoff_decisions evidence is available. State that quantified/documented trade-off analysis is not available.")

    if profile.get("resource_allocation_evidence_available"):
        instructions.append("- Use climate capex and climate opex as resource-allocation evidence.")

    if profile.get("high_carbon_exposure_available"):
        instructions.append("- Use high-carbon sector exposure metrics.")
    if profile.get("fossil_fuel_exposure_available"):
        instructions.append("- Use fossil-fuel exposure metrics.")

    if profile.get("climate_opportunities_available"):
        instructions.append("- Describe the concrete climate opportunities supplied in evidence.")
        if profile.get("quantified_opportunity_estimates_available"):
            instructions.append("- Use opportunity revenue estimates cautiously and identify them as estimates with confidence levels.")
    else:
        instructions.append("- No climate-opportunity evidence is available. Do not invent opportunities.")

    if profile.get("scenario_analysis_available"):
        instructions.append("- Explain scenario families, framework, horizons, assumptions and key modelled outputs.")
        instructions.append("- Explicitly cover scenario inputs/assumptions: carbon price, temperature outcomes, technology readiness, macroeconomic assumptions where available, renewable energy share where available, methodology notes and scope of analysis.")
    else:
        instructions.append("- Scenario analysis is unavailable. Do not invent scenario results.")

    if profile.get("resilience_evidence_available"):
        instructions.append("- Describe resilience only within supplied scenario-specific assumptions and boundaries.")
        instructions.append("- Cover adaptation-capacity evidence using available financial resources, portfolio flexibility mechanisms and current/planned investment evidence. Do not invent asset redeployment/decommissioning capacity.")

    if profile.get("targets_available"):
        if profile.get("full_targets_table_available"):
            instructions.append("- Full target records are available; use only fields explicitly provided.")
        elif profile.get("target_summary_available"):
            instructions.append("- Only target_summary is available. Use target type, scope, status, year, framework and progress only; do not invent baselines, validation bodies, milestones, gross/net status or planned credits.")
    else:
        instructions.append("- Target evidence is unavailable. Do not invent target information.")

    instructions.append("- For transition-plan disclosure, use available updated transition plan/net-zero target decision evidence and target frameworks/progress only. If key assumptions or dependencies are not available, state that boundary.")

    feedback_block = ""
    if judge_feedback:
        feedback_block = f"""
JUDGE FEEDBACK TO ADDRESS:
{judge_feedback}

REVISION POLICY:
- Fix omissions using available evidence.
- Preserve accurate evidence-boundary statements where evidence is unavailable.
- Never invent evidence to satisfy the judge.
""".strip()

    return f"""
{STRATEGY_REQUIREMENTS}

BANK:
{evidence['bank']['bank_name']} ({evidence['bank']['bank_id']})

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DATA-AWARE WRITING INSTRUCTIONS:
{chr(10).join(instructions)}

COMPACT STRATEGY EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

IMPORTANT DATA BOUNDARIES:
- Strategy payload tables available: {evidence.get('payload_profile', {}).get('top_level_keys')}
- Full targets table available: {evidence.get('payload_profile', {}).get('full_targets_table_available')}
- Target summary available: {evidence.get('payload_profile', {}).get('target_summary_available')}
- Target instruction: {boundaries.get('target_instruction')}
- Business model impacts available: {boundaries.get('business_model_detail_available')}
- Trade-off evidence available: {boundaries.get('tradeoff_evidence_available')}
- Scenario outputs are modelled estimates: {boundaries.get('scenario_outputs_are_modelled_estimates')}
- Opportunity impacts are estimates: {boundaries.get('opportunity_impacts_are_estimates')}

GENERAL WRITING RULES:
- Do not claim that opportunities are guaranteed revenue.
- Do not claim that scenario losses are actual losses.
- Do not claim that the bank is generally resilient.
- Do not claim overall Paris alignment.
- Use exact figures from compact evidence.
- Mention evidence boundaries only once in the relevant subsection.

{feedback_block}

Write the complete Strategy section only.
""".strip()


STRATEGY_JUDGE_SYSTEM = """
You are a strict sustainability-reporting judge for an IFRS S1/S2-aligned bank climate Strategy disclosure.

You evaluate whether the Strategy section is:
- supported by compact evidence;
- aligned with the Strategy disclosure checklist;
- transparent about missing evidence;
- free from unsupported claims about targets, opportunities, scenarios, business model, value chain, Paris alignment and resilience.

Return valid JSON only.
All checklist values must be JSON booleans true or false, not strings such as "true" or "false".
""".strip()


def build_strategy_judge_prompt(draft: str, evidence: dict, deterministic_checks: dict | None = None) -> str:
    profile = evidence.get("availability_profile", {})
    deterministic_checks = deterministic_checks or {}

    return f"""
Evaluate the Strategy draft against the compact evidence, availability profile and deterministic pre-checks.

DRAFT:
{draft}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic_checks, indent=2, ensure_ascii=False)}

COMPACT STRATEGY EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
1. Penalise claims that contradict evidence, invent unsupported information, or overstate estimates.
2. Penalise omission when material evidence is available but not used.
3. Do not demand unavailable evidence. Correctly disclosed boundaries are acceptable but lower completeness.
4. Verify use of physical and transition risks, time horizons, scenario evidence, value-chain map, opportunities, portfolio metrics and resource allocation.
5. If full targets table is unavailable, the draft must not invent baselines, validation bodies, milestones, gross/net status or planned credits.
6. If business_model_impacts are unavailable, the draft must not claim detailed business-model impact analysis beyond available lending/value-chain/financial channels.
7. If strategy_tradeoff_decisions are unavailable, the draft must not invent trade-off analysis.
8. Opportunity revenue impacts must be described as estimates, not guaranteed future revenue.
9. Scenario financial outputs must be described as modelled estimates, not actual losses.
10. Scenario assumptions should cover framework, scenario types/names, horizons, carbon price, temperature outcomes, technology readiness, macroeconomic/energy assumptions where available, methodology notes and scope.
11. Resilience claims must remain scenario-specific and bounded and should discuss adaptation capacity only from available evidence.
12. Transition-plan discussion must use available transition-plan/target evidence and state boundaries for missing assumptions/dependencies.
13. Overall Paris-alignment claims are not allowed unless directly supported and carefully bounded.
14. No visible IFRS paragraph references or bracketed tags should appear.
15. All checklist values must be JSON booleans true/false, not strings.

SCORING:
- 9–10: materially complete, directly evidenced and no material boundary required.
- 8: strong with one material boundary or minor evidence-use issue.
- 7: usable with multiple correctly disclosed boundaries.
- 6: revision required because available evidence was omitted, contradicted or overstated.
- 5 or below: major factual, evidence or overclaiming failures.

Return valid JSON only. Keep arrays concise:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "physical_and_transition_risks_used_correctly": <true/false>,
    "time_horizons_used_correctly": <true/false>,
    "business_model_handled_according_to_availability": <true/false>,
    "value_chain_used_correctly": <true/false>,
    "strategy_tradeoffs_handled_according_to_availability": <true/false>,
    "financial_effects_used_correctly": <true/false>,
    "resource_allocation_used_correctly": <true/false>,
    "high_carbon_and_fossil_exposure_used_if_available": <true/false>,
    "opportunities_used_correctly": <true/false>,
    "targets_handled_according_to_availability": <true/false>,
    "scenario_analysis_used_correctly": <true/false>,
    "scenario_assumptions_used_correctly": <true/false>,
    "transition_plan_handled_according_to_availability": <true/false>,
    "adaptation_capacity_handled_according_to_availability": <true/false>,
    "resilience_not_overclaimed": <true/false>,
    "paris_alignment_not_overclaimed": <true/false>,
    "modelled_estimates_not_overstated": <true/false>,
    "no_visible_ifrs_refs": <true/false>,
    "no_unsupported_strong_claims": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted>],
  "unsupported_claims": [<specific unsupported claims>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()


print("Strategy prompts ready")

In [ ]:
# ── STRATEGY EVALUATION MODE ───────────────────────────────
# Data-aware deterministic pre-checks are now active.
# They do not directly approve/reject the section or cap the score.
# Instead, their findings are passed to the GPT-5.2 Judge.

print("Strategy evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks")

In [ ]:
# ── STRATEGY LANGGRAPH NODES ────────────────────────────────

def extract_numbers(text: str) -> list[float]:
    nums = re.findall(r"\b\d+(?:\.\d+)?\b", text.replace(",", ""))
    return [float(n) for n in nums]

def number_present(text: str, expected: float, tolerance: float = 0.2) -> bool:
    nums = extract_numbers(text)
    return any(abs(n - expected) <= tolerance for n in nums)

def run_strategy_deterministic_checks(draft: str, evidence: dict) -> dict:
    text = draft or ""
    text_l = text.lower()
    profile = evidence.get("availability_profile", {})

    required_headings = [
        "#### Climate-related risks and opportunities",
        "#### Effects on business model and value chain",
        "#### Effects on strategy and decision-making",
        "#### Financial effects and resource allocation",
        "#### Climate resilience and scenario analysis",
        "#### Strategy limitations and evidence boundaries",
    ]

    missing_headings = [h for h in required_headings if h not in text]

    warnings = []
    failures = []

    if missing_headings:
        failures.append({"check": "missing_required_headings", "details": missing_headings})

    if re.search(r"IFRS\s*S?[12]?\s*§|§\s*\d|\[IFRS", text):
        failures.append({"check": "visible_ifrs_references", "details": "Visible IFRS references/tags found."})

    unsupported_strong_phrases = [
        "fully resilient",
        "guarantees",
        "guaranteed",
        "proves resilience",
        "fully paris aligned",
        "overall paris aligned",
        "no material risk",
    ]
    found_strong = [p for p in unsupported_strong_phrases if p in text_l]
    if found_strong:
        failures.append({"check": "unsupported_strong_strategy_language", "details": found_strong})

    if profile.get("climate_opportunities_available") and "opportun" not in text_l:
        warnings.append({"check": "opportunities_may_be_omitted", "details": "Climate opportunity evidence exists but opportunity wording is not detected."})

    if profile.get("quantified_opportunity_estimates_available"):
        if "estimate" not in text_l and "estimated" not in text_l:
            failures.append({"check": "opportunity_estimates_not_labelled", "details": "Opportunity revenue impacts should be described as estimates."})

    if profile.get("scenario_financial_outputs_available"):
        estimate_terms = ["modelled", "modeled", "scenario", "estimate", "projected"]
        if not any(term in text_l for term in estimate_terms):
            failures.append({"check": "scenario_outputs_not_labelled_as_estimates", "details": "Scenario financial outputs should be labelled as modelled estimates."})

    if profile.get("value_chain_detail_available") and "value chain" not in text_l:
        warnings.append({"check": "value_chain_may_be_omitted", "details": "Value-chain evidence exists but value-chain wording is not detected."})

    if profile.get("high_carbon_exposure_available") and "high-carbon" not in text_l and "high carbon" not in text_l:
        warnings.append({"check": "high_carbon_exposure_may_be_omitted", "details": "High-carbon exposure metrics exist but may be omitted."})

    if profile.get("fossil_fuel_exposure_available") and "fossil" not in text_l:
        warnings.append({"check": "fossil_fuel_exposure_may_be_omitted", "details": "Fossil-fuel exposure metrics exist but may be omitted."})

    if profile.get("resource_allocation_evidence_available"):
        if not number_present(text, 476.95, tolerance=0.2):
            failures.append({"check": "missing_climate_capex", "details": "Climate capex of EUR 476.95 million is available and should be used."})
        if not number_present(text, 219.32, tolerance=0.2):
            failures.append({"check": "missing_climate_opex", "details": "Climate opex of EUR 219.32 million is available and should be used."})

    if profile.get("target_summary_available") and not profile.get("full_targets_table_available"):
        unsupported_target_terms = [
            "baseline year",
            "baseline value",
            "validation body",
            "third-party validated",
            "gross target",
            "net target",
            "interim milestone",
            "planned carbon credits",
        ]
        found_target_terms = [p for p in unsupported_target_terms if p in text_l]
        if found_target_terms:
            failures.append({"check": "full_target_details_invented", "details": found_target_terms})

    if not profile.get("strategy_tradeoff_evidence_available"):
        if ("trade-off" in text_l or "tradeoff" in text_l) and not any(term in text_l for term in ["not available", "not documented", "does not provide", "no quantified"]):
            failures.append({"check": "strategy_tradeoff_overclaimed", "details": "Trade-off language appears without available trade-off evidence or limitation."})

    if not profile.get("business_model_detail_available"):
        if "detailed business model" in text_l and not any(term in text_l for term in ["limited", "not available", "not detailed"]):
            failures.append({"check": "business_model_detail_overclaimed", "details": "Detailed business-model impact evidence is absent."})

    if profile.get("scenario_assumptions_available"):
        assumption_terms = ["carbon price", "temperature", "technology readiness", "methodology", "scope of analysis"]
        missing_terms = [term for term in assumption_terms if term not in text_l]
        if missing_terms:
            warnings.append({"check": "scenario_assumption_terms_may_be_missing", "details": missing_terms})

    if profile.get("resilience_evidence_available"):
        if "resilience" in text_l and not any(term in text_l for term in ["scenario", "under", "within", "assumption"]):
            failures.append({"check": "resilience_not_scenario_bounded", "details": "Resilience wording should be scenario-specific and bounded."})

    return {
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


def strategy_writer_node(state: StrategyState) -> StrategyState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_strategy_writer_prompt(state["evidence"], judge_feedback=feedback)
    draft = call_writer_llm(
        system_prompt=STRATEGY_WRITER_SYSTEM,
        user_prompt=prompt,
    )

    print(f"\n{'='*50}")
    print(f"STRATEGY WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {**state, "draft": draft.strip(), "status": "judging"}


def strategy_judge_node(state: StrategyState) -> StrategyState:
    draft = state["draft"]
    deterministic_checks = run_strategy_deterministic_checks(draft, state["evidence"])

    prompt = build_strategy_judge_prompt(
        draft,
        state["evidence"],
        deterministic_checks=deterministic_checks,
    )
    judge_result = call_judge_llm_json(
        system_prompt=STRATEGY_JUDGE_SYSTEM,
        user_prompt=prompt,
    )
    judge_result = normalize_json_booleans(judge_result)

    judge_result.setdefault("approved", False)
    judge_result.setdefault(
        "approval_status",
        "approved" if judge_result.get("approved") else "revision_required",
    )
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})
    judge_result["deterministic_prechecks"] = deterministic_checks

    print("\nSTRATEGY JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))

    approved = bool(judge_result.get("approved"))
    status = "approved" if approved else "revising"

    return {
        **state,
        "judge_result": judge_result,
        "status": status,
    }


STRATEGY_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Strategy section using only:
- the supplied compact Strategy evidence;
- the availability profile;
- the data-aware deterministic pre-checks; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- Clearly distinguish estimates, scenarios and actual financial effects.
- Keep target details within available evidence.
- Keep the exact six-subsection structure.
- Do not add visible IFRS paragraph references.
- Return only the complete revised Strategy section.
""".strip()


def strategy_reviser_node(state: StrategyState) -> StrategyState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}

    judge = state.get("judge_result", {})
    issues = judge.get("required_fixes", [])
    checklist = judge.get("checklist", {})
    deterministic = judge.get("deterministic_prechecks", {})
    false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]

    revision_prompt = f"""
STRICT STRATEGY REQUIREMENTS:
{STRATEGY_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic, indent=2, ensure_ascii=False)}

COMPACT STRATEGY EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- Preserve or improve accurate boundary statements where evidence is unavailable.
- Never invent missing opportunities, trade-offs, business-model impacts, value-chain effects, target details, scenario assumptions, financial impacts, or resilience conclusions.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete revised Strategy section.
""".strip()

    revised_draft = call_reviser_llm(
        system_prompt=STRATEGY_REVISER_SYSTEM,
        user_prompt=revision_prompt,
    )

    new_revision_count = state["revision_count"] + 1
    print(f"\nStrategy revised with GPT-4.1 | revision {new_revision_count}")

    return {
        **state,
        "draft": revised_draft.strip(),
        "revision_count": new_revision_count,
        "status": "judging",
    }


def strategy_finalize_node(state: StrategyState) -> StrategyState:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)

    print(f"\n{'='*50}")
    print("FINALIZED STRATEGY")
    print(f"  Status: {'APPROVED' if approved else 'MAX REVISIONS REACHED'}")
    print(f"  Final score: {judge.get('overall_score')}/10")
    print(f"  Revisions: {state['revision_count']}")
    print(f"{'='*50}")

    return {
        **state,
        "final_section": state["draft"],
        "status": "approved" if approved else "failed",
    }


def strategy_should_continue(state: StrategyState) -> str:
    judge = state.get("judge_result", {})
    if judge.get("approved", False):
        return "finalize"

    if state.get("revision_count", 0) >= state.get("max_revisions", 1):
        return "finalize"

    return "reviser"

In [ ]:

# ── BUILD AND COMPILE STRATEGY GRAPH ────────────────────────
strategy_builder = StateGraph(StrategyState)

strategy_builder.add_node("writer",   strategy_writer_node)
strategy_builder.add_node("judge",    strategy_judge_node)
strategy_builder.add_node("reviser",  strategy_reviser_node)
strategy_builder.add_node("finalize", strategy_finalize_node)

strategy_builder.add_edge(START, "writer")
strategy_builder.add_edge("writer", "judge")
strategy_builder.add_conditional_edges(
    "judge",
    strategy_should_continue,
    {
        "finalize": "finalize",
        "reviser": "reviser",
    }
)
strategy_builder.add_edge("reviser", "judge")
strategy_builder.add_edge("finalize", END)

strategy_graph = strategy_builder.compile()
print("Strategy graph compiled")


In [ ]:

# ── RUN STRATEGY SECTION ────────────────────────────────────
strategy_initial_state: StrategyState = {
    "bank_name":      bank_name,
    "evidence":       strategy_evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  1,  # one GPT-4.1 revision pass if the GPT-5.2 judge rejects the draft
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {},
}

print(f"Starting strategy generation for: {bank_name}\n")
strategy_result = strategy_graph.invoke(strategy_initial_state)


In [ ]:

# ── STRATEGY OUTPUT ─────────────────────────────────────────
print("\n" + "="*60)
print("FINAL STRATEGY JUDGE RESULT")
print("="*60)
print(json.dumps(strategy_result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("STRATEGY SECTION")
print("="*60)
print(strategy_result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "strategy_BANK01.md", "w", encoding="utf-8") as f:
    f.write(strategy_result["final_section"])

with open(output_dir / "strategy_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "strategy",
        "status":         strategy_result["status"],
        "approval_status": strategy_result["judge_result"].get("approval_status"),
        "raw_score":      strategy_result["judge_result"].get("raw_score_before_caps"),
        "final_score":    strategy_result["judge_result"].get("overall_score"),
        "score_cap_reason": strategy_result["judge_result"].get("score_cap_reason"),
        "revisions":      strategy_result["revision_count"],
        "approved":       strategy_result["judge_result"].get("approved"),
        "checklist":      strategy_result["judge_result"].get("checklist"),
        "issues":         strategy_result["judge_result"].get("main_issues"),
        "available_evidence_omitted": strategy_result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims": strategy_result["judge_result"].get("unsupported_claims"),
        "correctly_disclosed_evidence_boundaries": strategy_result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_strategy_payload_path": str(RAW_STRATEGY_PATH),
        "compact_strategy_evidence_path": str(COMPACT_STRATEGY_PATH),
    }, f, indent=2, ensure_ascii=False)

print("\nSaved to outputs/strategy_BANK01.md")

print(f"Raw Strategy payload saved to: {RAW_STRATEGY_PATH}")
print(f"Compact Strategy evidence saved to: {COMPACT_STRATEGY_PATH}")
